# 04 · 最终模型与提交

**最终方案**（实验 15，提交文件 `submission_final_v4_final.csv`，五折交叉验证 75.966）：

| 步骤 | 内容 |
|---|---|
| 特征 | 02 的 ALL3，按模型不同（57 / 73 / 78 列） |
| 模型 | LightGBM、XGBoost、CatBoost、lgb_rel、cb_rel、cb_gl、TabM、sklearn HGB（03） |
| 验证 | 5 折 × 3 个种子（42、7、2024，每个种子一套不同的五折），外层训练部分内留 10% 早停；每个模型的 OOF 取种子平均 |
| 测试预测 | 树模型在全部 105,000 行上重训：轮数 = CV 平均最佳轮数 × 1.2，cb_rel、cb_gl 用学习率 0.03。TabM 和 HGB 在 95% / 全部数据上重训（内部早停）。多个种子平均 |
| 融合 | 全局 NNLS；用五折嵌套给出诚实的分数 |
| 第二层 | 双胞胎（01）+ S3 同一运动员残差平滑（05），按「性别 + 国籍 + 州」分两半交叉拟合 |

它由三个实验逐步完成，后一个复用前一个的结果：
- 实验 11：6 个模型；
- 实验 12：更多全量重训种子；
- 实验 15：加入 cb_gl 与 HGB 的三种子版本。

| notebook | 内容 |
|---|---|
| **01_data_exploration** | 读取与核对原始数据、探索分析、记录精度与「双胞胎」、数据质量检查、固定五折 |
| **02_feature_engineering** | 四轮特征（记录精度、跨地参赛、比赛背景、运动员历史与生涯、基于成绩的特征）及五折消融（实验 07、09、10、14） |
| **03_model_selection** | 早期基线、CatBoost 参数、其他模型与相对力量目标、堆叠方式、目标变量工程、模型大比较（实验 08、13） |
| **04_final_model** | 最终集成：8 个模型 × 5 折 × 3 种子 + 全量重训、NNLS 融合、双胞胎 + 同一运动员平滑、提交文件（实验 11、12、15） |
| **05_error_analysis** | 最终残差、同一运动员与双胞胎结构、第二层做法比较、数据清洗实验、未采用的方法、交叉验证与公开榜 |

原始数据在 `data/`；结果保存在 `artifacts/` 下与 notebook 同名的目录。「实验 07–15」是实验的原始编号，代码注释和 RESULTS.md 沿用这些编号。

**训练开关**：默认全部为 `False`，只读取 `artifacts/` 中已保存的结果并生成表格和图，Colab 普通 CPU 运行时即可，几分钟跑完。

把某一项改为 `True` 并选择 **T4 GPU** 运行时，会按原协议重新训练对应的实验。每个模型训练完立即写入 `artifacts/`，断线后从头运行会跳过已完成的部分。**一次只打开一个开关**（不同实验的代码共用变量名），跑完后重启运行时再打开下一个。

In [ ]:
TRAIN = {
    'six_models': False,     # 实验 11：6 个模型 × 5 折 × 3 种子 + 全量重训 + 数据清洗实验（GPU，约 2.5 小时）
    'more_seeds': False,     # 实验 12：cb_rel 学习率检查 + 全量重训再加 5 个种子 + TabM 95% 重训（GPU，约 1 小时）
    'eight_models': False,   # 实验 15：cb_gl、HGB 补齐三种子 + 全量重训 → 8 个模型的最终提交（GPU，约 1 小时）
}

In [ ]:
# 环境、路径与通用设置（5 本 notebook 相同）
from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import importlib
import importlib.metadata
import inspect
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.optimize import nnls
from sklearn.model_selection import KFold, train_test_split

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')


def find_project_dir():
    candidates = []
    if os.environ.get('PL_PROJECT_DIR'):
        candidates.append(Path(os.environ['PL_PROJECT_DIR']))
    candidates.append(Path('/content/drive/MyDrive/Colab Notebooks/performances-en-powerlifting'))
    here = Path.cwd().resolve()
    candidates += [here, *here.parents]
    for candidate in candidates:
        if (candidate / 'data' / 'train.csv').is_file() and (candidate / 'data' / 'test.csv').is_file():
            return candidate.expanduser().resolve()
    raise FileNotFoundError('找不到项目目录（应包含 data/train.csv 与 data/test.csv）；请设置环境变量 PL_PROJECT_DIR')


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / 'data'                 # 原始 CSV（只读）
ARTIFACTS = PROJECT_DIR / 'artifacts'           # 每本 notebook 的结果在与它同名的子目录
SEED = 42                                       # 固定五折：KFold(5, shuffle=True, random_state=42)
N_FOLDS = 5
INNER_STOP_FRACTION = 0.10                      # 外层训练部分内留 10% 早停
MAX_ROUNDS = 20000
MODELS = ['lgb', 'xgb', 'catboost']             # 特征消融用的三个模型（每折内的运行顺序：快的先跑）
RUN_ONLY = None                                 # 只跑部分变体时写成列表；None 表示全部
KEEP_DELTA = -0.05                              # kg：平均 ΔRMSE ≤ −0.05 且 ≥ 4/5 折更好才保留
EXPECTED_SHA256 = {'train.csv': '62319814116506fb95a51f1185dbacdb170ab055642ee4c347a30584093d0856',
                   'test.csv': '563c6fa29f9e535740ef1f09b25954f00fbe242c81c0e5a7343dc18aa962ba46'}
CATBOOST_VERSION = '1.2.10'
GPU_NAME = None
VERSIONS = {'numpy': np.__version__, 'pandas': pd.__version__, 'python': sys.version.split()[0]}
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

TRAIN = globals().get('TRAIN', {})              # 01 与 05 没有训练开关
if sum(bool(v) for v in TRAIN.values()) > 1:
    raise ValueError('一次只打开一个训练开关：不同实验的代码共用变量名。跑完一个、重启运行时后再打开下一个')
TRAINING = any(TRAIN.values())


def ensure_training_env():
    """只在训练时调用：安装固定版本的 CatBoost 与 pytabkit，导入训练用的库，确认 Colab 分配了 GPU。"""
    global catboost, lgb, xgb, torch, RealMLP_TD_Regressor, TabM_D_Regressor, PYTABKIT_ERROR, GPU_NAME, VERSIONS
    to_install = []
    try:
        installed_catboost = importlib.metadata.version('catboost')
    except importlib.metadata.PackageNotFoundError:
        installed_catboost = None
    if installed_catboost != CATBOOST_VERSION:
        if 'catboost' in sys.modules:
            raise RuntimeError('当前运行时已经导入另一版 CatBoost；请重启运行时后从头运行')
        to_install.append(f'catboost=={CATBOOST_VERSION}')
    try:
        importlib.metadata.version('pytabkit')
    except importlib.metadata.PackageNotFoundError:
        to_install.append('pytabkit')
    if to_install:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *to_install])
        importlib.invalidate_caches()
    import catboost
    import lightgbm as lgb
    import torch
    import xgboost as xgb
    try:
        from pytabkit import RealMLP_TD_Regressor, TabM_D_Regressor
        PYTABKIT_ERROR = None
    except Exception as exc:                    # 安装或导入失败时其余模型照常运行
        RealMLP_TD_Regressor = TabM_D_Regressor = None
        PYTABKIT_ERROR = repr(exc)
    warnings.filterwarnings('ignore', category=UserWarning)
    if catboost.__version__ != CATBOOST_VERSION:
        raise RuntimeError(f'需要 CatBoost {CATBOOST_VERSION}；实际为 {catboost.__version__}')
    if shutil.which('nvidia-smi') is None or not torch.cuda.is_available():
        raise RuntimeError('训练需要 GPU：请在 Colab「更改运行时类型」中选择 T4 GPU 后从头运行')
    GPU_NAME = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                              text=True, capture_output=True, check=False).stdout.strip().splitlines()[0]
    VERSIONS = {'catboost': catboost.__version__, 'xgboost': xgb.__version__, 'lightgbm': lgb.__version__,
                'torch': torch.__version__,
                'pytabkit': None if PYTABKIT_ERROR else importlib.metadata.version('pytabkit'),
                'numpy': np.__version__, 'pandas': pd.__version__, 'python': sys.version.split()[0]}
    print('GPU         :', GPU_NAME)
    print('版本         :', VERSIONS)
    if PYTABKIT_ERROR:
        print('pytabkit 不可用，RealMLP / TabM 会在预检中跳过：', PYTABKIT_ERROR)


print(f'项目：{PROJECT_DIR}')
print(f'Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}')
if TRAINING:
    print('打开的训练开关：', [k for k, v in TRAIN.items() if v])

## 0. 读取数据；打开训练开关时重新生成特征与模型定义（与 02、实验 11 相同的代码）

In [ ]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def rmse(a, b):
    a, b = np.asarray(a, dtype='float64'), np.asarray(b, dtype='float64')
    return float(np.sqrt(np.mean((a - b) ** 2)))


for name, expected in EXPECTED_SHA256.items():
    require(sha256_file(DATA_DIR / name) == expected, f'data/{name} 与原始数据不同；请检查数据文件')

TARGET = 'TotalKg'
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
RAW_NUM = ['Age', 'BodyweightKg', 'MeetYear']
RAW_FEATURES = [c for c in test.columns if c != 'id']
CAT = [c for c in RAW_FEATURES if c not in RAW_NUM]          # 不依赖 object / str dtype（pandas 2 与 3 都可用）
require(train.shape == (105000, 18) and test.shape == (45000, 17), '数据形状不符')
require(len(CAT) == 13 and all(pd.api.types.is_numeric_dtype(train[c]) for c in RAW_NUM)
        and not any(pd.api.types.is_numeric_dtype(train[c]) for c in CAT), '列分组不符')
require(not train[RAW_FEATURES].isna().any().any() and not test[RAW_FEATURES].isna().any().any(), '出现空值')
y = train[TARGET].to_numpy(dtype='float64')
n_train = len(train)

FOLD_IDS = np.zeros(n_train, dtype=int)
for k, (_, valid) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.arange(n_train)), start=1):
    FOLD_IDS[valid] = k
print(f'train {train.shape}  test {test.shape}；每折验证行数：{np.bincount(FOLD_IDS)[1:].tolist()}')

In [ ]:
if TRAINING:
    both = pd.concat([train[RAW_FEATURES], test[RAW_FEATURES]], ignore_index=True)

    # F0：v2 的体重级别特征
    weight_class = pd.to_numeric(both['WeightClassKg'].str.replace('+', '', regex=False), errors='coerce')
    F0_EXTRA = pd.DataFrame({
        'WeightClass_num': weight_class,
        'Is_SHW': both['WeightClassKg'].str.endswith('+').astype('float64'),
        'BW_minus_Class': both['BodyweightKg'] - weight_class,
    })

    # P：记录精度（树按阈值切数值，学不到「是不是 .5」「是不是整数」）
    age2 = (both['Age'] * 2).round()
    require(np.allclose(both['Age'] * 2, age2), 'Age 不是 0.5 的整数倍')
    bw = both['BodyweightKg']
    is_int = np.isclose(bw, bw.round())
    is_1dec = np.isclose(bw * 10, (bw * 10).round())
    is_half = is_1dec & ~is_int & np.isclose((bw * 10).round() % 10, 5)
    pounds = bw / 0.45359237
    P = pd.DataFrame({
        'age_half': (age2 % 2).astype('float64'),
        'bw_precision': np.select([is_int, is_half, is_1dec], [0.0, 1.0, 2.0], default=3.0),
        'bw_lb_int': (np.abs(pounds - pounds.round()) < 0.015).astype('float64'),
    })

    # T：跨地参赛（居住地未知时为 NaN）
    abroad = (both['Country'] != both['MeetCountry']).astype('float64').where(both['Country'] != 'Unknown')
    state_known = ((both['State'] != 'Unknown') & (both['MeetState'] != 'Unknown')
                   & (both['Country'] == both['MeetCountry']))
    other_state = (both['State'] != both['MeetState']).astype('float64').where(state_known)
    T = pd.DataFrame({'abroad': abroad, 'other_state': other_state})

    # M：比赛背景；「比赛」≈ 联盟 × 年份 × 比赛国 × 比赛州（train+test 无标签统计）
    meet = (both['Federation'] + '|' + both['MeetYear'].astype(str) + '|'
            + both['MeetCountry'] + '|' + both['MeetState'])


    def by_meet(values):
        return values.groupby(meet).transform('mean')


    M = pd.DataFrame({
        'meet_n': meet.map(meet.value_counts()).astype('float64'),
        'meet_sexeq_n': both.groupby([meet, both['Sex'], both['Equipment']])['Age'].transform('size').astype('float64'),
        'fed_year_n': both.groupby(['Federation', 'MeetYear'])['Age'].transform('size').astype('float64'),
        'meet_share_other_state': by_meet(other_state),
        'meet_share_abroad': by_meet(abroad),
        'meet_share_age_half': by_meet(P['age_half']),
        'meet_share_equipped': by_meet((both['Equipment'] != 'Raw').astype('float64')),
        'meet_share_male': by_meet((both['Sex'] == 'M').astype('float64')),
        'meet_mean_age': by_meet(both['Age']),
    })

    BLOCKS = {'P': P, 'T': T, 'M': M}
    VARIANTS = {'F0': [], 'P': ['P'], 'T': ['T'], 'M': ['M'], 'PT': ['P', 'T'], 'ALL': ['P', 'T', 'M']}
    for name, frame in {'F0': F0_EXTRA, **BLOCKS}.items():
        require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
        require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')

    _FRAMES = {}


    def frames_for(variant, model):
        """(train, test) 特征表。CatBoost 的类别为字符串；LGB / XGB 为 category（词表来自 train+test，无标签）。"""
        kind = 'str' if model == 'catboost' else 'category'
        if (variant, kind) not in _FRAMES:
            frame = pd.concat([both[RAW_NUM], F0_EXTRA] + [BLOCKS[b] for b in VARIANTS[variant]] + [both[CAT]], axis=1)
            require(frame.columns.is_unique, f'{variant} 列名重复')
            frame[CAT] = frame[CAT].astype(str).astype(object) if kind == 'str' else frame[CAT].astype('category')
            _FRAMES[(variant, kind)] = (frame.iloc[:n_train].reset_index(drop=True),
                                        frame.iloc[n_train:].reset_index(drop=True))
        return _FRAMES[(variant, kind)]


    print({v: frames_for(v, 'lgb')[0].shape[1] for v in VARIANTS})
    summary_new = pd.concat([P, T, M], axis=1)
    display(pd.DataFrame({'train_mean': summary_new.iloc[:n_train].mean(), 'test_mean': summary_new.iloc[n_train:].mean(),
                          'nan_share': summary_new.isna().mean()}).round(4))


    _FRAMES.clear()                     # 07 的变体表不再需要，释放内存
    CORE9 = ['Age', 'BodyweightKg', 'MeetYear', 'Sex', 'Federation', 'MeetCountry', 'MeetState', 'Country', 'State']
    sex, equipment = both['Sex'], both['Equipment']
    state = both['State'].where(both['State'] != 'Unknown')
    country = both['Country'].where(both['Country'] != 'Unknown')
    national = both['Federation'] + '|' + both['MeetYear'].astype(str) + '|' + both['MeetCountry']


    def counts(key):
        return key.map(key.value_counts()).astype('float64')


    def n_distinct(values, key):
        return values.groupby(key).transform('nunique').astype('float64')


    # M2：比赛背景加强（meet 与 by_meet 来自 07 的特征单元）
    M2 = pd.DataFrame({
        'meet_n_states': n_distinct(state, meet),
        'meet_n_countries': n_distinct(country, meet),
        'meet_n_wclass': n_distinct(both['WeightClassKg'], meet),
        'meet_share_tested': by_meet((both['Tested'] == 'Yes').astype('float64')),
        'meet_share_raw': by_meet((equipment == 'Raw').astype('float64')),
        'meet_share_multi': by_meet((equipment == 'Multi-ply').astype('float64')),
        'meet_share_bw2dec': by_meet((P['bw_precision'] == 3).astype('float64')),
        'meet_share_junior': by_meet((both['Age'] < 24).astype('float64')),
        'meet_share_master': by_meet((both['Age'] >= 40).astype('float64')),
        'meet_age_std': both['Age'].groupby(meet).transform('std'),
        'meet_sex_bw_mean': both['BodyweightKg'].groupby([meet, sex]).transform('mean'),
        'nat_n': counts(national),
        'nat_n_meetstates': n_distinct(both['MeetState'], national),
        'nat_n_states': n_distinct(state, national),
    })

    # D：组别背景
    division = meet + '|' + sex + '|' + equipment + '|' + both['WeightClassKg']
    age_division = meet + '|' + sex + '|' + equipment + '|' + both['AgeClass']
    exact_key = pd.Series(pd.util.hash_pandas_object(both[RAW_FEATURES], index=False).to_numpy())
    core_key = pd.Series(pd.util.hash_pandas_object(both[CORE9], index=False).to_numpy())
    D = pd.DataFrame({
        'div_n': counts(division),
        'div_bw_rank': both['BodyweightKg'].groupby(division).rank(pct=True),
        'div_age_rank': both['Age'].groupby(division).rank(pct=True),
        'agediv_n': counts(age_division),
        'dup_exact_n': counts(exact_key) - 1,
        'dup_core_n': counts(core_key) - 1,
    })

    # H：近似同一运动员的历史（出生年 = floor(MeetYear − Age)：x.5 年龄时恰为出生年，整数年龄可能差 1）
    valid = (both['Country'] != 'Unknown').to_numpy()
    birth = np.floor(both['MeetYear'] - both['Age'] + 1e-9).astype(int).astype(str)
    athlete = sex + '|' + both['Country'] + '|' + both['State'] + '|' + both['Federation'] + '|' + birth
    athlete0 = sex + '|' + both['Country'] + '|' + both['State'] + '|' + birth
    year = both['MeetYear'].astype('float64')
    H = pd.DataFrame({
        'ath_n': counts(athlete),
        'ath_n_years': n_distinct(year, athlete),
        'ath_years_since_first': year - year.groupby(athlete).transform('min'),
        'ath_years_to_last': year.groupby(athlete).transform('max') - year,
        'ath_n_equipment': n_distinct(equipment, athlete),
        'ath_bw_delta': both['BodyweightKg'] - both['BodyweightKg'].groupby(athlete).transform('mean'),
        'ath0_n': counts(athlete0),
        'ath0_n_feds': n_distinct(both['Federation'], athlete0),
    })
    H.loc[~valid] = np.nan


    # S：DOTS 系数（TotalKg = DOTS 分 × dots_scale）
    def dots_scale(sex_values, bw_values):
        s = np.asarray(sex_values).astype(str)
        bw = np.asarray(bw_values, dtype='float64')
        m, f = np.clip(bw, 40, 210), np.clip(bw, 40, 150)
        poly_m = -1.0930e-6 * m**4 + 7.391293e-4 * m**3 - 0.1918759221 * m**2 + 24.0900756 * m - 307.75076
        poly_f = -1.0706e-6 * f**4 + 5.158568e-4 * f**3 - 0.1126655495 * f**2 + 13.6175032 * f - 57.96288
        return np.where(s == 'M', poly_m, np.where(s == 'F', poly_f, 0.5 * (poly_m + poly_f))) / 500.0


    S = pd.DataFrame({'dots_scale': dots_scale(sex, both['BodyweightKg'])})

    # C：只给 CatBoost 的高基数 ID（有序目标统计编码）
    C = pd.DataFrame({'meet_id': meet, 'athlete_id': (athlete + '|' + equipment).where(valid, 'unknown')})

    BLOCKS2 = {'M2': M2, 'D': D, 'H': H, 'S': S}
    CB_ONLY = {'C': C}
    for name, frame in {**BLOCKS2, **CB_ONLY}.items():
        require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
    for name, frame in BLOCKS2.items():
        require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')
    require(S['dots_scale'].min() > 0.3, 'DOTS 系数异常')

    R1_BLOCKS = ['P', 'T', 'M']
    VARIANT_BLOCKS = {'R1': [], 'M2': ['M2'], 'D': ['D'], 'H': ['H'], 'S': ['S'], 'C': ['C']}
    MODELS_FOR = {'C': ['catboost']}      # C 只改变 CatBoost；融合时 LightGBM / XGBoost 用 R1
    SCREEN = [v for v in VARIANT_BLOCKS if RUN_ONLY is None or v in RUN_ONLY or v == 'R1']
    _FRAMES2 = {}


    def models_for(variant):
        return MODELS_FOR.get(variant, MODELS)


    def frames2(variant, model):
        """(train, test, 类别列)。R1 = F0 + P + T + M；C 块只加给 CatBoost。"""
        blocks = VARIANT_BLOCKS[variant]
        kind = 'str' if model == 'catboost' else 'category'
        key = (tuple(blocks), kind)
        if key not in _FRAMES2:
            nums = ([both[RAW_NUM], F0_EXTRA] + [BLOCKS[b] for b in R1_BLOCKS]
                    + [BLOCKS2[b] for b in blocks if b in BLOCKS2])
            cats = [both[CAT]] + ([CB_ONLY[b] for b in blocks if b in CB_ONLY] if kind == 'str' else [])
            frame = pd.concat(nums + cats, axis=1)
            require(frame.columns.is_unique, f'{variant} 列名重复')
            cat_cols = [c for part in cats for c in part.columns]
            if kind == 'str':
                frame[cat_cols] = frame[cat_cols].astype(str).astype(object)
            else:
                frame[cat_cols] = frame[cat_cols].astype('category')
            _FRAMES2[key] = (frame.iloc[:n_train].reset_index(drop=True),
                             frame.iloc[n_train:].reset_index(drop=True), cat_cols)
        return _FRAMES2[key]


    new_cols = pd.concat([M2, D, H, S], axis=1)
    display(pd.DataFrame({'train_mean': new_cols.iloc[:n_train].mean(), 'test_mean': new_cols.iloc[n_train:].mean(),
                          'nan_share': new_cols.isna().mean()}).round(4))
    print('C 块类别数：', {c: int(C[c].nunique()) for c in C.columns})
    print('各变体列数（LightGBM / XGBoost）：', {v: frames2(v, 'lgb')[0].shape[1] for v in VARIANT_BLOCKS if v != 'C'},
          '｜CatBoost C：', frames2('C', 'catboost')[0].shape[1])


    _FRAMES2.clear()                    # 09 的变体表不再需要，释放内存
    birth_num = np.floor(both['MeetYear'] - both['Age'] + 1e-9).astype(int)


    def fuzzy_count(prefix_key):
        """同前缀、出生年为 b−1、b、b+1 的行数（整数年龄推算的出生年可能差 1）。"""
        exact = prefix_key + '|' + birth_num.astype(str)
        table = exact.value_counts()
        total = exact.map(table).astype('float64')
        for d in (-1, 1):
            total = total + (prefix_key + '|' + (birth_num + d).astype(str)).map(table).fillna(0)
        return total


    # H2：运动员生涯（athlete、valid 来自 09 的特征单元；M、T 来自 07；M2 来自 09）
    bw = both['BodyweightKg']
    H2 = pd.DataFrame({
        'ath_max_meet_states': M2['meet_n_states'].groupby(athlete).transform('max'),
        'ath_mean_meet_states': M2['meet_n_states'].groupby(athlete).transform('mean'),
        'ath_max_meet_n': M['meet_n'].groupby(athlete).transform('max'),
        'ath_max_nat_states': M2['nat_n_states'].groupby(athlete).transform('max'),
        'ath_ever_abroad': T['abroad'].groupby(athlete).transform('max'),
        'ath_n_wclass': n_distinct(both['WeightClassKg'], athlete),
        'ath_bw_range': bw.groupby(athlete).transform('max') - bw.groupby(athlete).transform('min'),
        'ath_first_age': both['Age'].groupby(athlete).transform('min'),
        'ath_n_same_year': both.groupby([athlete, both['MeetYear']])['Age'].transform('size').astype('float64'),
        'ath_share_tested': (both['Tested'] == 'Yes').astype('float64').groupby(athlete).transform('mean'),
        'ath_n_fuzzy': fuzzy_count(sex + '|' + both['Country'] + '|' + both['State'] + '|' + both['Federation']),
        'ath0_n_fuzzy': fuzzy_count(sex + '|' + both['Country'] + '|' + both['State']),
    })
    H2.loc[~valid] = np.nan

    # E：比赛的经验结构
    experience = H['ath_years_since_first']
    E = pd.DataFrame({
        'meet_mean_ath_n': H['ath_n'].groupby(meet).transform('mean'),
        'meet_mean_ath_exp': experience.groupby(meet).transform('mean'),
        'meet_share_veterans': (experience >= 3).astype('float64').where(experience.notna()).groupby(meet).transform('mean'),
        'meet_share_newcomers': (H['ath_n'] == 1).astype('float64').where(H['ath_n'].notna()).groupby(meet).transform('mean'),
    })

    ROUND3 = {'H2': H2, 'E': E}
    for name, frame in ROUND3.items():
        require(len(frame) == len(both) and frame.index.equals(both.index), f'{name} 行数或索引不符')
        require(not np.isinf(frame.to_numpy(dtype='float64')).any(), f'{name} 含无穷大')
    NUMERIC_BLOCKS = {**BLOCKS, **BLOCKS2, **ROUND3}          # P T M（07）、M2 D H S（09）、H2 E（本轮）
    R2_BLOCKS = ['P', 'T', 'M', 'H']                          # 09 的 ALL2
    EXTRAS = {'lgb': [], 'xgb': ['M2', 'S', 'D'], 'catboost': ['M2', 'C']}   # PM：09 中各模型单独有效的块
    VARIANTS3 = {'R2': [], 'H2': ['H2'], 'E': ['E'], 'PM': ['PM']}
    SCREEN3 = [v for v in VARIANTS3 if RUN_ONLY is None or v in RUN_ONLY or v == 'R2']
    _FRAMES3 = {}


    def blocks_for(variant, model):
        out = list(R2_BLOCKS)
        for b in VARIANTS3[variant]:
            out += EXTRAS[model] if b == 'PM' else [b]
        return out


    def runs(model, variant):
        """该模型在这个变体上的特征与 R2 不同才需要训练（例如 PM 对 LightGBM 没有变化）。"""
        return variant == 'R2' or blocks_for(variant, model) != blocks_for('R2', model)


    def frames3(variant, model):
        """(train, test, 类别列)。"""
        blocks = blocks_for(variant, model)
        kind = 'str' if model == 'catboost' else 'category'
        key = (tuple(blocks), kind)
        if key not in _FRAMES3:
            nums = [both[RAW_NUM], F0_EXTRA] + [NUMERIC_BLOCKS[b] for b in blocks if b in NUMERIC_BLOCKS]
            cats = [both[CAT]] + ([CB_ONLY[b] for b in blocks if b in CB_ONLY] if kind == 'str' else [])
            frame = pd.concat(nums + cats, axis=1)
            require(frame.columns.is_unique, f'{variant} {model} 列名重复')
            cat_cols = [c for part in cats for c in part.columns]
            if kind == 'str':
                frame[cat_cols] = frame[cat_cols].astype(str).astype(object)
            else:
                frame[cat_cols] = frame[cat_cols].astype('category')
            _FRAMES3[key] = (frame.iloc[:n_train].reset_index(drop=True),
                             frame.iloc[n_train:].reset_index(drop=True), cat_cols)
        return _FRAMES3[key]


    new_cols = pd.concat([H2, E], axis=1)
    display(pd.DataFrame({'train_mean': new_cols.iloc[:n_train].mean(), 'test_mean': new_cols.iloc[n_train:].mean(),
                          'nan_share': new_cols.isna().mean()}).round(4))
    print({f'{v}/{m}': frames3(v, m)[0].shape[1] for v in VARIANTS3 for m in MODELS if runs(m, v)})
else:
    print("没有打开训练开关：不需要重新生成特征，下面只读取已保存的结果。")

In [ ]:
if TRAINING:
    ensure_training_env()
    SEEDS = [42, 7, 2024]                      # 每个种子：不同的五折划分、内层早停划分和模型随机性
    FINAL_MODELS = ['lgb', 'xgb', 'lgb_rel', 'catboost', 'cb_rel', 'tabm']   # 每折内的运行顺序：快的先跑
    FULL_REFIT = True                          # 树模型在全部训练行上重训
    REFIT_FACTOR = 1.2                         # 全量重训轮数 = CV 平均最佳轮数 × 1.2（数据多约 39%）

    # ---- 最终特征：10 的 ALL3（三个块都达标）----
    _FRAMES3.clear()
    VARIANTS3['ALL3'] = ['H2', 'E', 'PM']
    FAMILY = {'lgb': 'lgb', 'lgb_rel': 'lgb', 'tabm': 'lgb', 'xgb': 'xgb', 'catboost': 'catboost', 'cb_rel': 'catboost'}
    FRAMES = {m: frames3('ALL3', FAMILY[m]) for m in FINAL_MODELS}
    for m, (Xtr_m, Xte_m, cats_m) in FRAMES.items():
        print(f'{m:9s} {Xtr_m.shape[1]} 列（{len(cats_m)} 个类别列），块：{blocks_for("ALL3", FAMILY[m])}')


    LGB_PARAMS = dict(learning_rate=0.05, num_leaves=63, min_child_samples=40, subsample=0.8, subsample_freq=1,
                      colsample_bytree=0.8, cat_smooth=20, verbose=-1)
    XGB_PARAMS = dict(learning_rate=0.03, max_depth=8, min_child_weight=10.0, subsample=0.8, colsample_bytree=0.7,
                      reg_lambda=1.0, tree_method='hist', device='cuda', enable_categorical=True,
                      max_cat_to_onehot=1, eval_metric='rmse')
    CB_PARAMS = dict(learning_rate=0.05, depth=8, l2_leaf_reg=10.0, bootstrap_type='Bernoulli', subsample=0.8,
                     border_count=128)
    TABM_PARAMS = dict(device='cuda', n_cv=1, n_refit=0, verbosity=0, num_emb_type='pwl')
    TABM_FALLBACK = dict(device='cuda', n_cv=1, n_refit=0, verbosity=0)
    PATIENCE = {'lgb': 200, 'lgb_rel': 200, 'xgb': 200, 'catboost': 300, 'cb_rel': 300}
    GBDT = [m for m in FINAL_MODELS if m != 'tabm']

    SCALE = dots_scale(train['Sex'], train['BodyweightKg'])           # dots_scale 来自 09 的特征单元
    SCALE_TEST = dots_scale(test['Sex'], test['BodyweightKg'])
    require(min(SCALE.min(), SCALE_TEST.min()) > 0.3, 'DOTS 系数异常')

    X_NN, XT_NN, CATS_NN = FRAMES['tabm']
    NUM_NN = [c for c in X_NN.columns if c not in CATS_NN]
    NAN_NN = [c for c in NUM_NN if X_NN[c].isna().any() or XT_NN[c].isna().any()]


    def target_and_weight(model, rows):
        if model.endswith('_rel'):
            return y[rows] / SCALE[rows], SCALE[rows] ** 2       # 相对力量误差 × scale = kg 误差
        return y[rows], None


    def gbdt_fit(model, seed, Xa, ya, wa, Xs=None, ys=None, ws=None, rounds=MAX_ROUNDS):
        """Xs 为 None 时不早停（全量重训）。返回 (模型, 最佳轮数)。"""
        cats = FRAMES[model][2]
        if model in ('lgb', 'lgb_rel'):
            m = lgb.LGBMRegressor(n_estimators=rounds, random_state=seed, n_jobs=-1, **LGB_PARAMS)
            if Xs is None:
                m.fit(Xa, ya, sample_weight=wa)
                return m, rounds
            m.fit(Xa, ya, sample_weight=wa, eval_set=[(Xs, ys)], eval_sample_weight=None if ws is None else [ws],
                  callbacks=[lgb.early_stopping(PATIENCE[model], verbose=False)])
            return m, int(m.best_iteration_ or rounds)
        if model == 'xgb':
            if Xs is None:
                m = xgb.XGBRegressor(n_estimators=rounds, random_state=seed, **XGB_PARAMS)
                m.fit(Xa, ya, verbose=False)
                return m, rounds
            m = xgb.XGBRegressor(n_estimators=rounds, random_state=seed, early_stopping_rounds=PATIENCE['xgb'],
                                 **XGB_PARAMS)
            m.fit(Xa, ya, eval_set=[(Xs, ys)], verbose=False)
            return m, int(m.best_iteration) + 1
        common = dict(iterations=rounds, loss_function='RMSE', eval_metric='RMSE', random_seed=seed, verbose=0,
                      task_type='GPU', devices='0', allow_writing_files=False, **CB_PARAMS)
        if Xs is None:
            m = catboost.CatBoostRegressor(**common)
            m.fit(catboost.Pool(Xa, ya, cat_features=cats, weight=wa))
            return m, rounds
        m = catboost.CatBoostRegressor(od_type='Iter', od_wait=PATIENCE[model], **common)
        m.fit(catboost.Pool(Xa, ya, cat_features=cats, weight=wa),
              eval_set=catboost.Pool(Xs, ys, cat_features=cats, weight=ws), use_best_model=True)
        return m, int(m.get_best_iteration()) + 1


    def gbdt_predict(model, m, best, X):
        if model in ('lgb', 'lgb_rel'):
            return m.predict(X, num_iteration=best)
        if model == 'xgb':
            return m.predict(X, iteration_range=(0, best))
        return m.predict(X)


    def nn_frames(*frames):
        """数值缺失用第一个表（外层训练部分）的中位数填补并加缺失标记；类别保持 category。"""
        median = frames[0][NUM_NN].median()
        out = []
        for frame in frames:
            z = frame.copy()
            for c in NAN_NN:
                z[f'{c}_isna'] = z[c].isna().astype('float64')
            z[NUM_NN] = z[NUM_NN].fillna(median).fillna(0.0)
            out.append(z.reset_index(drop=True))
        return out


    def tabm_fit_predict(seed, a, s, v, Xt, overrides=None):
        require(TabM_D_Regressor is not None, f'pytabkit 不可用：{PYTABKIT_ERROR}')
        params = dict(TABM_PARAMS, random_state=seed, **(overrides or {}))
        unknown = set(params) - set(inspect.signature(TabM_D_Regressor.__init__).parameters)
        require(not unknown, f'TabM 不支持参数 {sorted(unknown)}')
        Za, Zs, Zv, Zt = nn_frames(X_NN.iloc[a], X_NN.iloc[s], X_NN.iloc[v], Xt)
        mu, sd = y[a].mean(), y[a].std()
        m = TabM_D_Regressor(**params)
        m.fit(Za, (y[a] - mu) / sd, Zs, (y[s] - mu) / sd, cat_col_names=[c for c in Za.columns if c in CATS_NN])
        p_val = np.asarray(m.predict(Zv), dtype='float64').reshape(-1) * sd + mu
        p_test = np.asarray(m.predict(Zt), dtype='float64').reshape(-1) * sd + mu
        del m
        gc.collect()
        torch.cuda.empty_cache()
        return p_val, p_test


    def fit_fold(model, seed, a, s, v, test_rows=None, rounds=MAX_ROUNDS, overrides=None):
        """a / s / v：内层拟合 / 内层早停 / 外层验证 的训练行号。返回 (验证预测, 测试预测, 最佳轮数或 None)。"""
        Xtr, Xte, _ = FRAMES[model]
        Xt = Xte if test_rows is None else Xte.iloc[test_rows]
        scale_t = SCALE_TEST if test_rows is None else SCALE_TEST[test_rows]
        if model == 'tabm':
            p_val, p_test = tabm_fit_predict(seed, a, s, v, Xt, overrides)
            return p_val, p_test, None
        ya, wa = target_and_weight(model, a)
        ys, ws = target_and_weight(model, s)
        if wa is not None:
            norm = wa.mean()
            wa, ws = wa / norm, ws / norm
        m, best = gbdt_fit(model, seed, Xtr.iloc[a], ya, wa, Xtr.iloc[s], ys, ws, rounds)
        p_val, p_test = gbdt_predict(model, m, best, Xtr.iloc[v]), gbdt_predict(model, m, best, Xt)
        if model.endswith('_rel'):
            p_val, p_test = np.asarray(p_val) * SCALE[v], np.asarray(p_test) * scale_t
        return np.asarray(p_val, dtype='float64'), np.asarray(p_test, dtype='float64'), best


    def fit_full(model, seed, rounds, test_rows=None):
        """全部训练行、固定轮数、不早停。返回测试预测。"""
        Xtr, Xte, _ = FRAMES[model]
        Xt = Xte if test_rows is None else Xte.iloc[test_rows]
        scale_t = SCALE_TEST if test_rows is None else SCALE_TEST[test_rows]
        rows = np.arange(n_train)
        ya, wa = target_and_weight(model, rows)
        if wa is not None:
            wa = wa / wa.mean()
        m, _ = gbdt_fit(model, seed, Xtr, ya, wa, rounds=rounds)
        p_test = gbdt_predict(model, m, rounds, Xt)
        if model.endswith('_rel'):
            p_test = np.asarray(p_test) * scale_t
        return np.asarray(p_test, dtype='float64')
else:
    print("没有打开训练开关：不需要模型定义。")

In [ ]:
# 只读的结果工具：读取训练时保存的 summary_*.csv 和逐折预测，不训练
def read_summary(path, index_col=0):
    path = Path(path)
    if not path.is_file():
        print(f'缺少 {path.relative_to(PROJECT_DIR)}：请确认 artifacts/ 已从 Drive 同步，或打开对应的训练开关')
        return None
    return pd.read_csv(path, index_col=index_col)


def show(table, columns=None, digits=3):
    if table is not None:
        shown = table if columns is None else table[[c for c in columns if c in table.columns]]
        display(shown.round(digits))


def plot_fold_deltas(table, baseline, title):
    """配对比较表（列 mean_delta、d_fold1–5）：柱 = 五折平均 ΔRMSE，点 = 单折，虚线 = 保留门槛 −0.05。"""
    if table is None:
        return
    rows = table.drop(index=baseline, errors='ignore').sort_values('mean_delta', ascending=False)
    folds = rows[[f'd_fold{k}' for k in range(1, N_FOLDS + 1)]].to_numpy(dtype='float64')
    colours = ['#4c72b0' if v <= KEEP_DELTA else '#c44e52' if v > 0 else '#999999' for v in rows['mean_delta']]
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(rows) + 1.6))
    ax.barh(range(len(rows)), rows['mean_delta'], color=colours)
    for i in range(len(rows)):
        ax.scatter(folds[i], np.full(folds.shape[1], i), color='black', s=12, zorder=3)
    ax.axvline(0, color='black', lw=0.8)
    ax.axvline(KEEP_DELTA, color='#4c72b0', lw=0.8, ls='--')
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels(rows.index.tolist())
    ax.set_xlabel(f'RMSE change vs {baseline} (kg); bar = 5-fold mean, dots = single folds')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def read_folds(directory, stem):
    """一个模型五折的 (OOF, 五折平均测试预测)；缺任何一折时返回 None。"""
    oof, test_pred = np.full(n_train, np.nan), np.zeros(len(test))
    for k in range(1, N_FOLDS + 1):
        path = Path(directory) / f'{stem}__fold{k}.npz'
        if not path.is_file():
            return None
        saved = np.load(path)
        oof[saved['valid_idx']] = saved['val']
        test_pred += saved['test'] / N_FOLDS
    return (oof, test_pred) if not np.isnan(oof).any() else None

## 1. 6 个模型 × 5 折 × 3 种子 + 全量重训（实验 11）

- LightGBM、XGBoost 用 v2 参数，CatBoost 用 C2。
- `cb_rel`、`lgb_rel` 的目标 = TotalKg ÷ DOTS 系数，样本权重 = 系数²，预测后乘回 kg。
- TabM：pytabkit 默认参数 + 分段线性数值嵌入；数值缺失用外层训练部分的中位数填补并加缺失标记，目标先标准化。

测试预测有三种：
- 折平均：每个模型 15 个折模型平均；
- 全量重训：树模型在全部训练行上重训，每个种子一次；
- 混合：两者各半。

同一个训练开关还包括数据清洗实验，结果在 05。

In [ ]:
if TRAIN['six_models']:
    ensure_training_env()
    # 实验 11 的设置（结果目录：原 artifacts/final_v1）
    OUTPUT_DIR = ARTIFACTS / '04_final_model' / 'v1_six_models'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    RUN_CLEANING_TEST = True                   # 数据清洗实验（只作报告证据，结果在 05）
    # ---- 预检：每个模型在 3,000 行上训练几轮；树模型再在全部行上做 30 轮的全量重训 ----
    rng = np.random.default_rng(SEED)
    probe = rng.choice(n_train, 3000, replace=False)
    pa, ps, pv = probe[:2000], probe[2000:2500], probe[2500:]
    failed = {}
    for model in FINAL_MODELS:
        attempts = ['main', 'fallback'] if model == 'tabm' else ['main']
        for attempt in attempts:
            if attempt == 'fallback':
                TABM_PARAMS = dict(TABM_FALLBACK)
            started = time.perf_counter()
            try:
                p_val, p_test, _ = fit_fold(model, SEED, pa, ps, pv, test_rows=np.arange(500), rounds=30,
                                            overrides={'n_epochs': 3})
                require(len(p_val) == len(pv) and len(p_test) == 500, '预测长度不符')
                require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), '预测含非有限值')
                if model in GBDT and FULL_REFIT:
                    p_full = fit_full(model, SEED, 30, test_rows=np.arange(500))
                    require(np.isfinite(p_full).all() and len(p_full) == 500, '全量重训预测无效')
                failed.pop(model, None)
                print(f'预检通过：{model:9s}（{attempt}）RMSE@少量轮 {rmse(y[pv], p_val):.1f}，'
                      f'用时 {time.perf_counter() - started:.0f}s')
                break
            except Exception as exc:
                failed[model] = repr(exc)
                print(f'预检失败：{model}（{attempt}）→ {exc!r}')
    RUN_MODELS = [m for m in FINAL_MODELS if m not in failed]
    print('将运行：', RUN_MODELS, '｜跳过：', failed or '无')
    require('catboost' in RUN_MODELS, 'CatBoost 预检失败，请检查环境')

    # ---- 五折 × 种子 ----
    CONFIG = {'purpose': 'final ensemble: ALL3 per-model features, 5-fold x seeds, inner 10% early stop, full refit',
              'seeds': SEEDS, 'n_folds': N_FOLDS, 'inner_stop_fraction': INNER_STOP_FRACTION,
              'refit_factor': REFIT_FACTOR, 'final_models': FINAL_MODELS, 'data_sha256': EXPECTED_SHA256}
    config_path = OUTPUT_DIR / 'config.json'
    if config_path.is_file():
        require(json.loads(config_path.read_text(encoding='utf-8')) == CONFIG,
                f'{OUTPUT_DIR.name} 已有不同配置的结果；改动配置时请改 EXPERIMENT_ID')
    else:
        config_path.write_text(json.dumps(CONFIG, ensure_ascii=False, indent=1), encoding='utf-8')


    def folds_for(seed):
        ids = np.zeros(n_train, dtype=int)
        for k, (_, valid) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=seed).split(np.arange(n_train)), start=1):
            ids[valid] = k
        return ids


    FOLDS_BY_SEED = {s: folds_for(s) for s in SEEDS}
    if SEED in FOLDS_BY_SEED:
        require(np.array_equal(FOLDS_BY_SEED[SEED], FOLD_IDS), '种子 42 的五折与 07–10 不一致')


    def spec_of(model):
        params = {'lgb': LGB_PARAMS, 'lgb_rel': LGB_PARAMS, 'xgb': XGB_PARAMS, 'catboost': CB_PARAMS,
                  'cb_rel': CB_PARAMS, 'tabm': TABM_PARAMS}[model]
        return {'model': model, 'columns': list(FRAMES[model][0].columns), 'cat_cols': list(FRAMES[model][2]),
                'params': dict(params), 'target': 'dots_relative_weighted' if model.endswith('_rel') else 'kg'}


    def save_result(stem, arrays, info):
        np.savez(OUTPUT_DIR / f'{stem}.npz', **arrays)
        info = {**info, 'gpu': GPU_NAME, 'versions': VERSIONS, 'finished_at_utc': datetime.now(timezone.utc).isoformat()}
        (OUTPUT_DIR / f'{stem}.json').write_text(json.dumps(info, ensure_ascii=False, indent=1), encoding='utf-8')
        return info


    def load_cached(stem, spec):
        json_path, npz_path = OUTPUT_DIR / f'{stem}.json', OUTPUT_DIR / f'{stem}.npz'
        if json_path.is_file() and npz_path.is_file():
            saved = json.loads(json_path.read_text(encoding='utf-8'))
            require(saved['spec'] == spec, f'{stem} 的定义已改变；请改 EXPERIMENT_ID')
            return saved
        return None


    def run_cv(model, seed, fold):
        stem, spec = f'{model}__s{seed}__fold{fold}', spec_of(model)
        saved = load_cached(stem, spec)
        if saved is not None:
            return saved, 'cache'
        ids = FOLDS_BY_SEED[seed]
        fit_idx, valid_idx = np.flatnonzero(ids != fold), np.flatnonzero(ids == fold)
        inner_fit, inner_stop = train_test_split(fit_idx, test_size=INNER_STOP_FRACTION, random_state=seed)
        started = time.perf_counter()
        p_val, p_test, best = fit_fold(model, seed, inner_fit, inner_stop, valid_idx)
        require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), f'{stem} 预测含非有限值')
        info = save_result(stem, {'valid_idx': valid_idx, 'val': p_val, 'test': p_test},
                           {'model': model, 'seed': seed, 'fold': fold, 'spec': spec, 'best_iteration': best,
                            'rmse': rmse(y[valid_idx], p_val), 'seconds': time.perf_counter() - started})
        return info, f'{info["seconds"]:.0f}s'


    def run_full(model, seed):
        stem, spec = f'{model}__s{seed}__full', spec_of(model)
        saved = load_cached(stem, spec)
        if saved is not None:
            return saved, 'cache'
        bests = [json.loads((OUTPUT_DIR / f'{model}__s{seed}__fold{k}.json').read_text(encoding='utf-8'))['best_iteration']
                 for k in range(1, N_FOLDS + 1)]
        rounds = int(round(np.mean(bests) * REFIT_FACTOR))
        started = time.perf_counter()
        p_test = fit_full(model, seed, rounds)
        require(np.isfinite(p_test).all(), f'{stem} 预测含非有限值')
        info = save_result(stem, {'test': p_test}, {'model': model, 'seed': seed, 'spec': spec, 'rounds': rounds,
                                                    'cv_best_iterations': bests,
                                                    'seconds': time.perf_counter() - started})
        return info, f'{info["seconds"]:.0f}s'


    for seed in SEEDS:
        seed_started = time.perf_counter()
        for fold in range(1, N_FOLDS + 1):
            for model in RUN_MODELS:
                info, how = run_cv(model, seed, fold)
                best = info['best_iteration']
                print(f'种子 {seed:4d} fold {fold}  {model:9s} RMSE {info["rmse"]:.3f}'
                      + (f'  最佳轮数 {best}' if best else '') + f'  ({how})')
        if FULL_REFIT:
            for model in RUN_MODELS:
                if model in GBDT:
                    info, how = run_full(model, seed)
                    print(f'种子 {seed:4d} 全量重训 {model:9s} 轮数 {info["rounds"]}  ({how})')
        print(f'==== 种子 {seed} 完成，本次运行用时 {(time.perf_counter() - seed_started) / 60:.1f} 分钟 ====')


    def load_seed(model, seed):
        oof, test_pred = np.full(n_train, np.nan), np.zeros(len(test))
        for fold in range(1, N_FOLDS + 1):
            path = OUTPUT_DIR / f'{model}__s{seed}__fold{fold}.npz'
            if not path.is_file():
                return None
            saved = np.load(path)
            oof[saved['valid_idx']] = saved['val']
            test_pred += saved['test'] / N_FOLDS
        require(not np.isnan(oof).any(), f'{model} 种子 {seed} 的 OOF 不完整')
        return oof, test_pred


    def fold_rmse(pred, ids=FOLD_IDS):
        return pd.Series({k: rmse(y[ids == k], pred[ids == k]) for k in range(1, N_FOLDS + 1)})


    PER_MODEL = {}
    rows = []
    for model in FINAL_MODELS:
        loaded = {s: load_seed(model, s) for s in SEEDS}
        loaded = {s: v for s, v in loaded.items() if v is not None}
        if not loaded:
            continue
        oof = np.mean([o for o, _ in loaded.values()], axis=0)
        fold_avg = np.mean([t for _, t in loaded.values()], axis=0)
        fulls = [np.load(OUTPUT_DIR / f'{model}__s{s}__full.npz')['test'] for s in loaded
                 if (OUTPUT_DIR / f'{model}__s{s}__full.npz').is_file()]
        full = np.mean(fulls, axis=0) if fulls else fold_avg          # TabM 没有全量重训，用折平均
        PER_MODEL[model] = {'oof': oof, 'fold_avg': fold_avg, 'full': full}
        rows.append({'model': model, 'seeds': list(loaded), 'full_refits': len(fulls),
                     **{f'oof_seed{s}': rmse(y, o) for s, (o, _) in loaded.items()},
                     'oof_seed_avg': rmse(y, oof)})
    model_table = pd.DataFrame(rows).set_index('model').sort_values('oof_seed_avg')
    display(model_table.round(3))
    model_table.to_csv(OUTPUT_DIR / 'summary_models.csv')

    NAMES = list(PER_MODEL)
    P = np.column_stack([PER_MODEL[n]['oof'] for n in NAMES])
    WEIGHTS, _ = nnls(P, y)
    NESTED = np.zeros(n_train)
    for k in range(1, N_FOLDS + 1):                       # 诚实分数：每一折的权重只用其他四折拟合
        fit, va = FOLD_IDS != k, FOLD_IDS == k
        w_k, _ = nnls(P[fit], y[fit])
        NESTED[va] = P[va] @ w_k
    BLEND_TEST = {kind: np.column_stack([PER_MODEL[n][kind] for n in NAMES]) @ WEIGHTS for kind in ['fold_avg', 'full']}
    BLEND_TEST['hybrid'] = 0.5 * (BLEND_TEST['fold_avg'] + BLEND_TEST['full'])
    print('NNLS 权重：', dict(zip(NAMES, WEIGHTS.round(3))), '｜权重和', round(float(WEIGHTS.sum()), 4))
    print(f'融合 OOF：权重同一份 OOF 拟合 {rmse(y, P @ WEIGHTS):.3f}，嵌套（诚实）{rmse(y, NESTED):.3f}')
    print('三种测试预测之间的平均绝对差：',
          {f'{a}-{b}': round(float(np.abs(BLEND_TEST[a] - BLEND_TEST[b]).mean()), 2)
           for a, b in [('fold_avg', 'full'), ('fold_avg', 'hybrid'), ('full', 'hybrid')]})
    pd.DataFrame({'model': NAMES, 'weight': WEIGHTS}).to_csv(OUTPUT_DIR / 'summary_weights.csv', index=False)


    CORE = CORE9


    def row_keys(frame, columns):
        return pd.util.hash_pandas_object(frame[columns], index=False).to_numpy()


    def mate_stats(train_keys, query_keys=None):
        table = pd.DataFrame({'k': train_keys, 'y': y}).groupby('k')['y'].agg(['sum', 'count'])
        if query_keys is None:                      # 训练行：留一
            keys = pd.Series(train_keys)
            n = keys.map(table['count']).to_numpy() - 1
            s = keys.map(table['sum']).to_numpy() - y
        else:                                       # 测试行：全部训练行
            keys = pd.Series(query_keys)
            n = keys.map(table['count']).fillna(0).to_numpy()
            s = keys.map(table['sum']).fillna(0).to_numpy()
        return n, np.where(n > 0, s / np.maximum(n, 1), np.nan)


    def twins(query_exact=None, query_core=None):
        n_ex, m_ex = mate_stats(row_keys(train, RAW_FEATURES), query_exact)
        n_co, m_co = mate_stats(row_keys(train, CORE), query_core)
        level = np.select([n_ex > 0, n_co > 0], ['exact', 'core'], default='none')
        return level, np.where(level == 'exact', m_ex, m_co)


    def fit_alpha(mask, pred, mate):
        d = mate[mask] - pred[mask]
        denom = float(np.sum(d * d))
        return float(np.clip(np.sum((y[mask] - pred[mask]) * d) / denom, 0, 1)) if denom > 0 else 0.0


    def apply_twins(pred, mate, level, alphas):
        out = pred.copy()
        for lv, a in alphas.items():
            m = level == lv
            out[m] = pred[m] + a * (mate[m] - pred[m])
        return out


    LEVEL, MATE = twins()
    T_LEVEL, T_MATE = twins(row_keys(test, RAW_FEATURES), row_keys(test, CORE))
    HALF = (pd.util.hash_pandas_object(train[['Sex', 'Country', 'State']], index=False).to_numpy() % 2).astype(int)
    TR, TE = train.reset_index(drop=True), test.reset_index(drop=True)


    def text_key(frame, columns):
        out = frame[columns[0]].astype(str)
        for c in columns[1:]:
            out = out + '|' + frame[c].astype(str)
        return out


    BY_TR = np.floor(TR['MeetYear'] - TR['Age'] + 1e-9).astype(int)
    BY_TE = np.floor(TE['MeetYear'] - TE['Age'] + 1e-9).astype(int)
    YR_TR, YR_TE = TR['MeetYear'].astype(int), TE['MeetYear'].astype(int)
    PREFIX = {'A0': ['Sex', 'Country', 'State'], 'A1': ['Sex', 'Country', 'State', 'Federation'],
              'A2': ['Sex', 'Country', 'State', 'Federation', 'Equipment'],
              'A3': ['Sex', 'Country', 'State', 'Federation', 'Equipment', 'WeightClassKg']}
    SHRINK = {'A0': 5.0, 'A1': 5.0, 'A2': 3.0, 'A3': 2.0}
    PFX_TR = {k: text_key(TR, c) for k, c in PREFIX.items()}
    PFX_TE = {k: text_key(TE, c) for k, c in PREFIX.items()}
    MEET_COLS = ['Federation', 'MeetYear', 'MeetCountry', 'MeetState', 'Sex', 'Equipment']
    MEET_TR, MEET_TE = text_key(TR, MEET_COLS), text_key(TE, MEET_COLS)
    VALID_TR = (TR['Country'] != 'Unknown').to_numpy(dtype='float64')
    VALID_TE = (TE['Country'] != 'Unknown').to_numpy(dtype='float64')


    def sums(keys, values):
        table = pd.DataFrame({'k': keys.to_numpy(), 'v': values}).groupby('k')['v'].agg(['sum', 'count'])
        return table['sum'], table['count']


    def look(table, keys):
        s, n = table
        return keys.map(s).fillna(0).to_numpy(dtype='float64'), keys.map(n).fillna(0).to_numpy(dtype='float64')


    def f_exact(k, r):
        ktr, kte = PFX_TR[k] + '|' + BY_TR.astype(str), PFX_TE[k] + '|' + BY_TE.astype(str)
        table = sums(ktr, r)
        s, n = look(table, ktr)
        st, nt = look(table, kte)
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


    def f_fuzzy(k, r):
        table = sums(PFX_TR[k] + '|' + BY_TR.astype(str), r)
        s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
        for d in (-1, 0, 1):
            a, b = look(table, PFX_TR[k] + '|' + (BY_TR + d).astype(str))
            s, n = s + a, n + b
            a, b = look(table, PFX_TE[k] + '|' + (BY_TE + d).astype(str))
            st, nt = st + a, nt + b
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


    def f_decay(k, r, tau=3.0, max_gap=8):
        base_tr = PFX_TR[k] + '|' + BY_TR.astype(str) + '|'
        base_te = PFX_TE[k] + '|' + BY_TE.astype(str) + '|'
        table = sums(base_tr + YR_TR.astype(str), r)
        s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
        for d in range(-max_gap, max_gap + 1):
            w = float(np.exp(-abs(d) / tau))
            a, b = look(table, base_tr + (YR_TR + d).astype(str))
            s, n = s + w * a, n + w * b
            a, b = look(table, base_te + (YR_TE + d).astype(str))
            st, nt = st + w * a, nt + w * b
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])     # d = 0 时自己权重为 1，已扣除


    def f_meet(r, lam=20.0):
        table = sums(MEET_TR, r)
        s, n = look(table, MEET_TR)
        st, nt = look(table, MEET_TE)
        return (s - r) / (n - 1 + lam), st / (nt + lam)


    def s3_features(r):
        cols_tr, cols_te = [], []
        for builder in (f_exact, f_fuzzy, f_decay):
            for k in PREFIX:
                a, b = builder(k, r)
                cols_tr.append(a * VALID_TR)
                cols_te.append(b * VALID_TE)
        a, b = f_meet(r)
        cols_tr.append(a)
        cols_te.append(b)
        return np.column_stack(cols_tr), np.column_stack(cols_te)


    # 第二层只依赖训练行的 OOF（嵌套融合），对三种测试预测使用同一组 alpha / 系数
    alphas = {lv: fit_alpha(LEVEL == lv, NESTED, MATE) for lv in ['exact', 'core']}
    p1 = apply_twins(NESTED, MATE, LEVEL, alphas)
    p1_cf = NESTED.copy()
    for h in (0, 1):
        a_h = {lv: fit_alpha((HALF != h) & (LEVEL == lv), NESTED, MATE) for lv in ['exact', 'core']}
        m = HALF == h
        p1_cf[m] = apply_twins(NESTED[m], MATE[m], LEVEL[m], a_h)
    r1 = y - p1
    F, FT = s3_features(r1)
    p2_cf = p1_cf.copy()
    for h in (0, 1):
        fit, m = HALF != h, HALF == h
        beta_h = np.linalg.lstsq(F[fit], r1[fit], rcond=None)[0]
        p2_cf[m] = p1_cf[m] + F[m] @ beta_h
    BETA = np.linalg.lstsq(F, r1, rcond=None)[0]
    STAGE2_ADJ = FT @ BETA
    stage_table = pd.DataFrame([{'step': '融合（嵌套）', 'oof_rmse': rmse(y, NESTED)},
                                {'step': '+ 双胞胎（交叉拟合）', 'oof_rmse': rmse(y, p1_cf)},
                                {'step': '+ S3 平滑（交叉拟合）', 'oof_rmse': rmse(y, p2_cf)}]).set_index('step')
    display(stage_table.round(3))
    stage_table.to_csv(OUTPUT_DIR / 'summary_stage2.csv')
    print('alpha：', {k: round(v, 3) for k, v in alphas.items()}, '｜测试中有双胞胎的行：', int((T_LEVEL != 'none').sum()),
          '｜S3 平均调整幅度：', round(float(np.abs(STAGE2_ADJ).mean()), 2), 'kg')

    for kind, pred in BLEND_TEST.items():
        final = apply_twins(pred, T_MATE, T_LEVEL, alphas) + STAGE2_ADJ
        for stage, values in [('base', pred), ('final', final)]:
            path = OUTPUT_DIR / f'submission_final_{kind}_{stage}.csv'
            pd.DataFrame({'id': test['id'], TARGET: np.clip(values, 0, None)}).to_csv(path, index=False)
            print('写出', path.name)
    np.save(OUTPUT_DIR / 'oof_nested_blend.npy', NESTED)
    np.save(OUTPUT_DIR / 'oof_final_crossfit.npy', p2_cf)


    if RUN_CLEANING_TEST:
        ratio = y / train['BodyweightKg'].to_numpy(dtype='float64')
        SUSPICIOUS = (np.where(train['Sex'].to_numpy() == 'M', ratio < 2.5, ratio < 1.8)
                      | (train['Age'].to_numpy(dtype='float64') < 5))
        print(f'疑似行：{int(SUSPICIOUS.sum())}（{SUSPICIOUS.mean():.2%}）')
        clean_rows = []
        for model in [m for m in ['lgb', 'xgb', 'catboost'] if m in RUN_MODELS]:
            if not all((OUTPUT_DIR / f'{model}__s{SEED}__fold{k}.npz').is_file() for k in range(1, N_FOLDS + 1)):
                print(f'{model}：种子 {SEED} 的五折还没跑完，跳过对照')
                continue
            base_folds, clean_folds, base_normal, clean_normal = [], [], [], []
            for fold in range(1, N_FOLDS + 1):
                stem, spec = f'clean_{model}__s{SEED}__fold{fold}', {**spec_of(model), 'cleaning': 'drop_suspicious'}
                saved = load_cached(stem, spec)
                ids = FOLDS_BY_SEED[SEED]
                fit_idx, valid_idx = np.flatnonzero(ids != fold), np.flatnonzero(ids == fold)
                if saved is None:
                    inner_fit, inner_stop = train_test_split(fit_idx, test_size=INNER_STOP_FRACTION, random_state=SEED)
                    inner_fit, inner_stop = inner_fit[~SUSPICIOUS[inner_fit]], inner_stop[~SUSPICIOUS[inner_stop]]
                    started = time.perf_counter()
                    p_val, _, best = fit_fold(model, SEED, inner_fit, inner_stop, valid_idx, test_rows=np.arange(10))
                    save_result(stem, {'valid_idx': valid_idx, 'val': p_val},
                                {'model': model, 'fold': fold, 'spec': spec, 'best_iteration': best,
                                 'rmse': rmse(y[valid_idx], p_val), 'seconds': time.perf_counter() - started})
                clean_val = np.load(OUTPUT_DIR / f'{stem}.npz')['val']
                base_val = np.load(OUTPUT_DIR / f'{model}__s{SEED}__fold{fold}.npz')['val']
                normal = ~SUSPICIOUS[valid_idx]
                base_folds.append(rmse(y[valid_idx], base_val))
                clean_folds.append(rmse(y[valid_idx], clean_val))
                base_normal.append(rmse(y[valid_idx][normal], base_val[normal]))
                clean_normal.append(rmse(y[valid_idx][normal], clean_val[normal]))
            delta = np.array(clean_folds) - np.array(base_folds)
            clean_rows.append({'model': model, 'base_all_rows': np.mean(base_folds), 'clean_all_rows': np.mean(clean_folds),
                               'delta_all_rows': delta.mean(), 'folds_better': int((delta < 0).sum()),
                               'base_normal_rows': np.mean(base_normal), 'clean_normal_rows': np.mean(clean_normal)})
        clean_table = pd.DataFrame(clean_rows).set_index('model')
        display(clean_table.round(3))
        clean_table.to_csv(OUTPUT_DIR / 'summary_cleaning_test.csv')
        print('delta_all_rows < 0 表示去掉疑似行后在全部验证行上更好；normal_rows 只看非疑似的验证行。')
    print('完成。')
else:
    print("训练开关关闭（TRAIN['six_models'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
def show_v1():
    directory = ARTIFACTS / '04_final_model' / 'v1_six_models'
    print('每个模型：三个种子各自的五折 OOF RMSE 与三种子平均：')
    show(read_summary(directory / 'summary_models.csv'))
    weights = read_summary(directory / 'summary_weights.csv')
    if weights is not None:
        print('NNLS 权重：', weights['weight'].round(3).to_dict())
    print('第二层（交叉拟合）：')
    show(read_summary(directory / 'summary_stage2.csv'))
    display(pd.DataFrame([('折平均：每个模型 15 个折模型的测试预测平均', 76.520),
                          ('混合：全量与折平均各半', 76.421),
                          ('全量重训：树模型在全部 105,000 行上重训 × 3 个种子，TabM 用折平均', 76.360)],
                         columns=['测试预测（都加双胞胎 + S3）', '公开榜']).set_index('测试预测（都加双胞胎 + S3）'))


show_v1()

**结论**：
- 三个种子平均让每个模型降约 0.4–0.8 kg；cb_rel 是最强的单模型（三种子 76.784，权重 0.45）。
- 融合 76.395 → 双胞胎 76.172 → S3 76.044。
- **全量重训比折平均好 0.16**（公开榜 76.360 对 76.520）：训练数据越多越好。之后的提交都用全量重训。

## 2. 更多全量重训种子 + TabM 近全量（实验 12）

- cb_rel、XGBoost、lgb_rel、LightGBM 的全量重训各加 5 个种子（共 8 个）。cb_rel 的新种子用学习率 0.03，学习率检查见 03。
- TabM 没有固定轮数可以照搬，改为在 95% 的数据上训练（留 5% 早停）× 3 个种子。
- 融合权重和第二层系数沿用实验 11（全量模型无法交叉验证）。

In [ ]:
if TRAIN['more_seeds']:
    ensure_training_env()
    # 实验 12 的设置（结果目录：原 artifacts/final_v2）
    OUTPUT_DIR = ARTIFACTS / '04_final_model' / 'v2_more_seeds'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    PREV_DIR = ARTIFACTS / '04_final_model' / 'v1_six_models'     # 实验 11 的结果
    PREV_SEEDS = [42, 7, 2024]
    EXTRA_SEEDS = [11, 23, 57, 99, 123]                          # 全量重训新增的种子
    EXTRA_MODELS = ['lgb', 'xgb', 'lgb_rel', 'cb_rel']           # 快的先跑
    TABM_FULL_SEEDS = [42, 7, 2024]
    TABM_HOLDOUT = 0.05                                          # TabM 近全量：留 5% 早停
    RUN_LR_CHECK = True
    if not (PREV_DIR / 'summary_weights.csv').is_file() or not (PREV_DIR / 'oof_nested_blend.npy').is_file():
        raise FileNotFoundError(f'缺少实验 11 的结果：{PREV_DIR}')
    CB_PARAMS_BASE = dict(CB_PARAMS)
    CB_PARAMS_LR03 = dict(CB_PARAMS_BASE, learning_rate=0.03)


    def with_cb(params, fn, *args, **kwargs):
        """临时替换 CatBoost 参数（gbdt_fit 在调用时读取全局 CB_PARAMS）。"""
        global CB_PARAMS
        saved = CB_PARAMS
        CB_PARAMS = params
        try:
            return fn(*args, **kwargs)
        finally:
            CB_PARAMS = saved


    def save_result(stem, arrays, info):
        np.savez(OUTPUT_DIR / f'{stem}.npz', **arrays)
        info = {**info, 'gpu': GPU_NAME, 'versions': VERSIONS, 'finished_at_utc': datetime.now(timezone.utc).isoformat()}
        (OUTPUT_DIR / f'{stem}.json').write_text(json.dumps(info, ensure_ascii=False, indent=1), encoding='utf-8')
        return info


    def load_cached(stem, spec):
        json_path, npz_path = OUTPUT_DIR / f'{stem}.json', OUTPUT_DIR / f'{stem}.npz'
        if json_path.is_file() and npz_path.is_file():
            saved = json.loads(json_path.read_text(encoding='utf-8'))
            require(saved['spec'] == spec, f'{stem} 的定义已改变；请改 EXPERIMENT_ID')
            return saved
        return None


    # 11 的结果必须完整：每个模型 3 个种子 × 5 折，树模型 3 个全量重训
    for model in FINAL_MODELS:
        for s in PREV_SEEDS:
            for k in range(1, N_FOLDS + 1):
                require((PREV_DIR / f'{model}__s{s}__fold{k}.npz').is_file(), f'11 缺少 {model} 种子 {s} 第 {k} 折')
            if model != 'tabm':
                require((PREV_DIR / f'{model}__s{s}__full.npz').is_file(), f'11 缺少 {model} 种子 {s} 的全量重训')

    # 预检：几秒内确认 cb_rel（学习率 0.03）和 TabM 近全量的代码路径可用
    rng = np.random.default_rng(SEED)
    probe = rng.choice(n_train, 3000, replace=False)
    pa, ps, pv = probe[:2000], probe[2000:2500], probe[2500:]
    p_val, p_test, _ = with_cb(CB_PARAMS_LR03, fit_fold, 'cb_rel', SEED, pa, ps, pv, test_rows=np.arange(500), rounds=30)
    require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), 'cb_rel 预检失败')
    p_full = with_cb(CB_PARAMS_LR03, fit_full, 'cb_rel', SEED, 30, test_rows=np.arange(500))
    require(np.isfinite(p_full).all(), 'cb_rel 全量重训预检失败')
    TABM_OK = True
    try:
        p_val, p_test = tabm_fit_predict(SEED, pa, ps, pv, XT_NN.iloc[:500], overrides={'n_epochs': 3})
        require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), 'TabM 预测无效')
    except Exception as exc:
        try:                                               # 与 11 相同：分段线性嵌入不可用时去掉
            TABM_PARAMS = dict(TABM_FALLBACK)
            p_val, p_test = tabm_fit_predict(SEED, pa, ps, pv, XT_NN.iloc[:500], overrides={'n_epochs': 3})
            print('TabM 改用不带嵌入的版本：', repr(exc))
        except Exception as exc2:
            TABM_OK = False
            print('TabM 预检失败，跳过 TabM 近全量：', repr(exc2))
    print('预检完成；TabM 近全量：', '可用' if TABM_OK else '跳过')


    USE_LR03 = False
    if RUN_LR_CHECK:
        lr_rows = []
        for fold in range(1, N_FOLDS + 1):
            stem = f'cb_rel_lr03__s{SEED}__fold{fold}'
            spec = {'model': 'cb_rel', 'params': dict(CB_PARAMS_LR03), 'columns': list(FRAMES['cb_rel'][0].columns)}
            info = load_cached(stem, spec)
            if info is None:
                fit_idx, valid_idx = np.flatnonzero(FOLD_IDS != fold), np.flatnonzero(FOLD_IDS == fold)
                inner_fit, inner_stop = train_test_split(fit_idx, test_size=INNER_STOP_FRACTION, random_state=SEED)  # = 11
                started = time.perf_counter()
                p_val, _, best = with_cb(CB_PARAMS_LR03, fit_fold, 'cb_rel', SEED, inner_fit, inner_stop, valid_idx,
                                         test_rows=np.arange(10))
                info = save_result(stem, {'valid_idx': valid_idx, 'val': p_val},
                                   {'spec': spec, 'fold': fold, 'best_iteration': best,
                                    'rmse': rmse(y[valid_idx], p_val), 'seconds': time.perf_counter() - started})
            base = json.loads((PREV_DIR / f'cb_rel__s{SEED}__fold{fold}.json').read_text(encoding='utf-8'))
            lr_rows.append({'fold': fold, 'lr_0.05': base['rmse'], 'lr_0.03': info['rmse'],
                            'delta': info['rmse'] - base['rmse'], 'best_iter_0.05': base['best_iteration'],
                            'best_iter_0.03': info['best_iteration'], 'minutes_0.03': info['seconds'] / 60})
            print(f'fold {fold}: lr 0.05 {base["rmse"]:.3f} → lr 0.03 {info["rmse"]:.3f}')
        lr_table = pd.DataFrame(lr_rows).set_index('fold')
        display(lr_table.round(3))
        lr_table.to_csv(OUTPUT_DIR / 'summary_lr_check.csv')
        USE_LR03 = bool(lr_table['delta'].mean() <= -0.02 and (lr_table['delta'] < 0).sum() >= 4)
    print('cb_rel 全量重训使用学习率', 0.03 if USE_LR03 else 0.05)


    def prev_best_iterations(model):
        return [json.loads((PREV_DIR / f'{model}__s{s}__fold{k}.json').read_text(encoding='utf-8'))['best_iteration']
                for s in PREV_SEEDS for k in range(1, N_FOLDS + 1)]


    def refit_plan(model):
        """(参数, 轮数)：轮数 = CV 平均最佳轮数 × REFIT_FACTOR。"""
        if model == 'cb_rel' and USE_LR03:
            bests = [json.loads((OUTPUT_DIR / f'cb_rel_lr03__s{SEED}__fold{k}.json').read_text(encoding='utf-8'))
                     ['best_iteration'] for k in range(1, N_FOLDS + 1)]
            return CB_PARAMS_LR03, int(round(np.mean(bests) * REFIT_FACTOR))
        return CB_PARAMS_BASE, int(round(np.mean(prev_best_iterations(model)) * REFIT_FACTOR))


    for seed in EXTRA_SEEDS:
        for model in EXTRA_MODELS:
            params, rounds = refit_plan(model)
            stem = f'{model}__s{seed}__full'
            spec = {'model': model, 'rounds': rounds, 'columns': list(FRAMES[model][0].columns),
                    'cb_params': dict(params) if model in ('catboost', 'cb_rel') else None}
            info = load_cached(stem, spec)
            if info is None:
                started = time.perf_counter()
                p_test = with_cb(params, fit_full, model, seed, rounds)
                require(np.isfinite(p_test).all(), f'{stem} 预测含非有限值')
                info = save_result(stem, {'test': p_test}, {'spec': spec, 'seed': seed,
                                                            'seconds': time.perf_counter() - started})
                how = f'{info["seconds"]:.0f}s'
            else:
                how = 'cache'
            print(f'种子 {seed:3d} 全量重训 {model:8s} 轮数 {rounds}  ({how})')

    if TABM_OK:
        for seed in TABM_FULL_SEEDS:
            stem = f'tabm__s{seed}__nearfull'
            spec = {'model': 'tabm', 'params': dict(TABM_PARAMS), 'holdout': TABM_HOLDOUT,
                    'columns': list(X_NN.columns)}
            info = load_cached(stem, spec)
            if info is None:
                fit_rows, stop_rows = train_test_split(np.arange(n_train), test_size=TABM_HOLDOUT, random_state=seed)
                started = time.perf_counter()
                p_stop, p_test = tabm_fit_predict(seed, fit_rows, stop_rows, stop_rows, XT_NN)
                require(np.isfinite(p_test).all(), f'{stem} 预测含非有限值')
                info = save_result(stem, {'test': p_test}, {'spec': spec, 'seed': seed,
                                                            'holdout_rmse': rmse(y[stop_rows], p_stop),
                                                            'seconds': time.perf_counter() - started})
            print(f'TabM 近全量 种子 {seed}：早停集 RMSE {info["holdout_rmse"]:.3f}（仅供参考，早停用过它）')


    weights = pd.read_csv(PREV_DIR / 'summary_weights.csv')
    NAMES, W = weights['model'].tolist(), weights['weight'].to_numpy(dtype='float64')
    NESTED = np.load(PREV_DIR / 'oof_nested_blend.npy')
    require(len(NESTED) == n_train and set(NAMES) <= set(FINAL_MODELS), '11 的权重或 OOF 不符')


    def fold_average(model):
        return np.mean([np.load(PREV_DIR / f'{model}__s{s}__fold{k}.npz')['test']
                        for s in PREV_SEEDS for k in range(1, N_FOLDS + 1)], axis=0)


    rows, PER_MODEL = [], {}
    for model in NAMES:
        if model == 'tabm':
            fold_avg = fold_average('tabm')
            near = [np.load(p)['test'] for p in sorted(OUTPUT_DIR.glob('tabm__s*__nearfull.npz'))]
            PER_MODEL[model] = {'seeds': fold_avg, 'seeds_tabmfull': np.mean(near, axis=0) if near else fold_avg}
            rows.append({'model': model, 'test_models': f'{len(near)} 个近全量（另一版用 15 个折模型）'})
        else:
            files = sorted(PREV_DIR.glob(f'{model}__s*__full.npz')) + sorted(OUTPUT_DIR.glob(f'{model}__s*__full.npz'))
            full = np.mean([np.load(p)['test'] for p in files], axis=0)
            PER_MODEL[model] = {'seeds': full, 'seeds_tabmfull': full}
            rows.append({'model': model, 'test_models': f'{len(files)} 个全量重训'})
    display(pd.DataFrame(rows).assign(weight=W).set_index('model'))

    BLEND_TEST = {f'v2_{kind}': np.column_stack([PER_MODEL[m][kind] for m in NAMES]) @ W
                  for kind in ['seeds', 'seeds_tabmfull']}
    prev_full = pd.read_csv(PREV_DIR / 'submission_final_full_base.csv')[TARGET].to_numpy()
    for kind, pred in BLEND_TEST.items():
        print(f'{kind}：与 11 全量版（未加第二层）的平均绝对差 {np.abs(np.clip(pred, 0, None) - prev_full).mean():.2f} kg')


    CORE = CORE9


    def row_keys(frame, columns):
        return pd.util.hash_pandas_object(frame[columns], index=False).to_numpy()


    def mate_stats(train_keys, query_keys=None):
        table = pd.DataFrame({'k': train_keys, 'y': y}).groupby('k')['y'].agg(['sum', 'count'])
        if query_keys is None:                      # 训练行：留一
            keys = pd.Series(train_keys)
            n = keys.map(table['count']).to_numpy() - 1
            s = keys.map(table['sum']).to_numpy() - y
        else:                                       # 测试行：全部训练行
            keys = pd.Series(query_keys)
            n = keys.map(table['count']).fillna(0).to_numpy()
            s = keys.map(table['sum']).fillna(0).to_numpy()
        return n, np.where(n > 0, s / np.maximum(n, 1), np.nan)


    def twins(query_exact=None, query_core=None):
        n_ex, m_ex = mate_stats(row_keys(train, RAW_FEATURES), query_exact)
        n_co, m_co = mate_stats(row_keys(train, CORE), query_core)
        level = np.select([n_ex > 0, n_co > 0], ['exact', 'core'], default='none')
        return level, np.where(level == 'exact', m_ex, m_co)


    def fit_alpha(mask, pred, mate):
        d = mate[mask] - pred[mask]
        denom = float(np.sum(d * d))
        return float(np.clip(np.sum((y[mask] - pred[mask]) * d) / denom, 0, 1)) if denom > 0 else 0.0


    def apply_twins(pred, mate, level, alphas):
        out = pred.copy()
        for lv, a in alphas.items():
            m = level == lv
            out[m] = pred[m] + a * (mate[m] - pred[m])
        return out


    LEVEL, MATE = twins()
    T_LEVEL, T_MATE = twins(row_keys(test, RAW_FEATURES), row_keys(test, CORE))
    HALF = (pd.util.hash_pandas_object(train[['Sex', 'Country', 'State']], index=False).to_numpy() % 2).astype(int)
    TR, TE = train.reset_index(drop=True), test.reset_index(drop=True)


    def text_key(frame, columns):
        out = frame[columns[0]].astype(str)
        for c in columns[1:]:
            out = out + '|' + frame[c].astype(str)
        return out


    BY_TR = np.floor(TR['MeetYear'] - TR['Age'] + 1e-9).astype(int)
    BY_TE = np.floor(TE['MeetYear'] - TE['Age'] + 1e-9).astype(int)
    YR_TR, YR_TE = TR['MeetYear'].astype(int), TE['MeetYear'].astype(int)
    PREFIX = {'A0': ['Sex', 'Country', 'State'], 'A1': ['Sex', 'Country', 'State', 'Federation'],
              'A2': ['Sex', 'Country', 'State', 'Federation', 'Equipment'],
              'A3': ['Sex', 'Country', 'State', 'Federation', 'Equipment', 'WeightClassKg']}
    SHRINK = {'A0': 5.0, 'A1': 5.0, 'A2': 3.0, 'A3': 2.0}
    PFX_TR = {k: text_key(TR, c) for k, c in PREFIX.items()}
    PFX_TE = {k: text_key(TE, c) for k, c in PREFIX.items()}
    MEET_COLS = ['Federation', 'MeetYear', 'MeetCountry', 'MeetState', 'Sex', 'Equipment']
    MEET_TR, MEET_TE = text_key(TR, MEET_COLS), text_key(TE, MEET_COLS)
    VALID_TR = (TR['Country'] != 'Unknown').to_numpy(dtype='float64')
    VALID_TE = (TE['Country'] != 'Unknown').to_numpy(dtype='float64')


    def sums(keys, values):
        table = pd.DataFrame({'k': keys.to_numpy(), 'v': values}).groupby('k')['v'].agg(['sum', 'count'])
        return table['sum'], table['count']


    def look(table, keys):
        s, n = table
        return keys.map(s).fillna(0).to_numpy(dtype='float64'), keys.map(n).fillna(0).to_numpy(dtype='float64')


    def f_exact(k, r):
        ktr, kte = PFX_TR[k] + '|' + BY_TR.astype(str), PFX_TE[k] + '|' + BY_TE.astype(str)
        table = sums(ktr, r)
        s, n = look(table, ktr)
        st, nt = look(table, kte)
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


    def f_fuzzy(k, r):
        table = sums(PFX_TR[k] + '|' + BY_TR.astype(str), r)
        s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
        for d in (-1, 0, 1):
            a, b = look(table, PFX_TR[k] + '|' + (BY_TR + d).astype(str))
            s, n = s + a, n + b
            a, b = look(table, PFX_TE[k] + '|' + (BY_TE + d).astype(str))
            st, nt = st + a, nt + b
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


    def f_decay(k, r, tau=3.0, max_gap=8):
        base_tr = PFX_TR[k] + '|' + BY_TR.astype(str) + '|'
        base_te = PFX_TE[k] + '|' + BY_TE.astype(str) + '|'
        table = sums(base_tr + YR_TR.astype(str), r)
        s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
        for d in range(-max_gap, max_gap + 1):
            w = float(np.exp(-abs(d) / tau))
            a, b = look(table, base_tr + (YR_TR + d).astype(str))
            s, n = s + w * a, n + w * b
            a, b = look(table, base_te + (YR_TE + d).astype(str))
            st, nt = st + w * a, nt + w * b
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])     # d = 0 时自己权重为 1，已扣除


    def f_meet(r, lam=20.0):
        table = sums(MEET_TR, r)
        s, n = look(table, MEET_TR)
        st, nt = look(table, MEET_TE)
        return (s - r) / (n - 1 + lam), st / (nt + lam)


    def s3_features(r):
        cols_tr, cols_te = [], []
        for builder in (f_exact, f_fuzzy, f_decay):
            for k in PREFIX:
                a, b = builder(k, r)
                cols_tr.append(a * VALID_TR)
                cols_te.append(b * VALID_TE)
        a, b = f_meet(r)
        cols_tr.append(a)
        cols_te.append(b)
        return np.column_stack(cols_tr), np.column_stack(cols_te)


    # 第二层只依赖训练行的 OOF（嵌套融合），对三种测试预测使用同一组 alpha / 系数
    alphas = {lv: fit_alpha(LEVEL == lv, NESTED, MATE) for lv in ['exact', 'core']}
    p1 = apply_twins(NESTED, MATE, LEVEL, alphas)
    p1_cf = NESTED.copy()
    for h in (0, 1):
        a_h = {lv: fit_alpha((HALF != h) & (LEVEL == lv), NESTED, MATE) for lv in ['exact', 'core']}
        m = HALF == h
        p1_cf[m] = apply_twins(NESTED[m], MATE[m], LEVEL[m], a_h)
    r1 = y - p1
    F, FT = s3_features(r1)
    p2_cf = p1_cf.copy()
    for h in (0, 1):
        fit, m = HALF != h, HALF == h
        beta_h = np.linalg.lstsq(F[fit], r1[fit], rcond=None)[0]
        p2_cf[m] = p1_cf[m] + F[m] @ beta_h
    BETA = np.linalg.lstsq(F, r1, rcond=None)[0]
    STAGE2_ADJ = FT @ BETA
    stage_table = pd.DataFrame([{'step': '融合（嵌套）', 'oof_rmse': rmse(y, NESTED)},
                                {'step': '+ 双胞胎（交叉拟合）', 'oof_rmse': rmse(y, p1_cf)},
                                {'step': '+ S3 平滑（交叉拟合）', 'oof_rmse': rmse(y, p2_cf)}]).set_index('step')
    display(stage_table.round(3))
    stage_table.to_csv(OUTPUT_DIR / 'summary_stage2.csv')
    print('alpha：', {k: round(v, 3) for k, v in alphas.items()}, '｜测试中有双胞胎的行：', int((T_LEVEL != 'none').sum()),
          '｜S3 平均调整幅度：', round(float(np.abs(STAGE2_ADJ).mean()), 2), 'kg')

    for kind, pred in BLEND_TEST.items():
        final = apply_twins(pred, T_MATE, T_LEVEL, alphas) + STAGE2_ADJ
        for stage, values in [('base', pred), ('final', final)]:
            path = OUTPUT_DIR / f'submission_final_{kind}_{stage}.csv'
            pd.DataFrame({'id': test['id'], TARGET: np.clip(values, 0, None)}).to_csv(path, index=False)
            print('写出', path.name)
    np.save(OUTPUT_DIR / 'oof_nested_blend.npy', NESTED)
    np.save(OUTPUT_DIR / 'oof_final_crossfit.npy', p2_cf)
else:
    print("训练开关关闭（TRAIN['more_seeds'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
def show_v2():
    v1 = ARTIFACTS / '04_final_model' / 'v1_six_models'
    v2 = ARTIFACTS / '04_final_model' / 'v2_more_seeds'
    counts = {m: len(list(v1.glob(f'{m}__s*__full.npz'))) + len(list(v2.glob(f'{m}__s*__full.npz')))
              for m in ['lgb', 'xgb', 'lgb_rel', 'catboost', 'cb_rel']}
    counts['tabm'] = len(list(v2.glob('tabm__s*__nearfull.npz')))
    print('每个模型用于测试预测的全量（TabM：95% 数据）重训模型个数：', counts)
    display(pd.DataFrame([('v2_seeds：全量重训共 8 个种子，TabM 折平均', 76.359),
                          ('v2_seeds_tabmfull：再加 TabM 95% 重训 × 3 个种子', 76.337)],
                         columns=['提交（权重与第二层沿用实验 11）', '公开榜']).set_index('提交（权重与第二层沿用实验 11）'))


show_v2()

## 3. 加入 sklearn HGB（实验 13 的融合部分）

03 的模型大比较中只有 HGB 达到加入门槛。在实验 12 的全部改进之上加入 HGB（五折平均），得到 `v3_plus`，公开榜 **76.334**。训练代码在 03 的模型比较开关里。

In [ ]:
print('实验 13：实验 12 的全部改进 + sklearn HGB（五折平均），融合与第二层（交叉拟合）：')
show(read_summary(ARTIFACTS / '03_model_selection' / 'model_compare' / 'summary_stage2.csv'))

## 4. 最终版：8 个模型（实验 15）

复用实验 11–14 的全部结果，只补缺的部分：

| 步骤 | 内容 |
|---|---|
| 1 | `cb_gl` 补种子 7、2024 的五折（种子 42 复用实验 14） |
| 2 | sklearn HGB 补种子 7、2024 的五折（种子 42 复用实验 13） |
| 3 | `cb_gl` 全量重训 × 5 个种子（学习率 0.03） |
| 4 | HGB 在全部数据上重训（内部留 10% 早停）× 3 个种子 |
| 5 | TabM 95% 重训再加 2 个种子（共 5 个） |
| 6 | 8 个模型 NNLS 融合 → 双胞胎 + S3 → 导出 → 自动检查 |

下面的结果表不是读取保存的数字，而是从逐折预测文件**重新计算**的；最后与保存的提交文件逐行核对。

In [ ]:
if TRAIN['eight_models']:
    ensure_training_env()
    # 实验 15 的设置（结果目录：原 artifacts/final_v3）
    OUTPUT_DIR = ARTIFACTS / '04_final_model' / 'v4_eight_models'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    FINAL_V1 = ARTIFACTS / '04_final_model' / 'v1_six_models'          # 实验 11
    FINAL_V2 = ARTIFACTS / '04_final_model' / 'v2_more_seeds'          # 实验 12
    COMPARE_DIR = ARTIFACTS / '03_model_selection' / 'model_compare'   # 实验 13
    ROUND4_DIR = ARTIFACTS / '02_feature_engineering' / 'round4'       # 实验 14
    PREV_SEEDS = [42, 7, 2024]
    GL_FULL_SEEDS = [42, 7, 2024, 11, 23]
    HGB_FULL_SEEDS = [42, 7, 2024]
    TABM_EXTRA_SEEDS = [11, 23]
    TABM_HOLDOUT = 0.05
    for directory in (FINAL_V1, FINAL_V2):
        if not directory.is_dir():
            raise FileNotFoundError(f'缺少之前的结果目录：{directory}')
    from sklearn.ensemble import HistGradientBoostingRegressor
    from sklearn.preprocessing import OrdinalEncoder


    def gl_scale(sex_values, equipment_values, bw_values):
        """IPF GL 系数的分母 / 100：总成绩 = GL 分 × gl_scale。Raw / Wraps 用无装备系数，其余用装备系数。"""
        sex = np.asarray(sex_values).astype(str)
        equipped = ~np.isin(np.asarray(equipment_values).astype(str), ['Raw', 'Wraps'])
        bw = np.maximum(np.asarray(bw_values, dtype='float64'), 30.0)
        coef = {('M', False): (1199.72839, 1025.18162, 0.00921), ('M', True): (1236.25115, 1449.21864, 0.01644),
                ('F', False): (610.32796, 1045.59282, 0.03048), ('F', True): (758.63878, 949.31382, 0.02435)}

        def curve(s, e):
            a, b, c = coef[(s, e)]
            return a - b * np.exp(-c * bw)

        male = np.where(equipped, curve('M', True), curve('M', False))
        female = np.where(equipped, curve('F', True), curve('F', False))
        return np.where(sex == 'M', male, np.where(sex == 'F', female, 0.5 * (male + female))) / 100.0


    GL = gl_scale(train['Sex'], train['Equipment'], train['BodyweightKg'])
    GL_TEST = gl_scale(test['Sex'], test['Equipment'], test['BodyweightKg'])
    require(min(GL.min(), GL_TEST.min()) > 0, 'GL 系数异常')
    SCALE_OF = {'cb_rel': (SCALE, SCALE_TEST), 'cb_gl': (GL, GL_TEST)}
    FRAMES['cb_gl'] = FRAMES['cb_rel']                    # 与 14 相同：CatBoost 族的 ALL3 特征
    PATIENCE['cb_gl'] = PATIENCE['cb_rel']
    CB_PARAMS_BASE = dict(CB_PARAMS)
    CB_PARAMS_LR03 = dict(CB_PARAMS_BASE, learning_rate=0.03)


    def with_cb(params, fn, *args, **kwargs):
        """临时替换 CatBoost 参数（gbdt_fit 在调用时读取全局 CB_PARAMS）。"""
        global CB_PARAMS
        saved = CB_PARAMS
        CB_PARAMS = params
        try:
            return fn(*args, **kwargs)
        finally:
            CB_PARAMS = saved


    def fit_matrix(model, a, s, v, seed, rounds=MAX_ROUNDS, test_rows=None):
        """与 14 相同：相对力量目标（按系数² 加权），返回 (外层验证预测, 测试预测, 最佳轮数)。"""
        Xtr, Xte, _ = FRAMES[model]
        Xt = Xte if test_rows is None else Xte.iloc[test_rows]
        scale, scale_test = SCALE_OF[model]
        scale_t = scale_test if test_rows is None else scale_test[test_rows]
        norm = (scale[a] ** 2).mean()
        m, best = gbdt_fit(model, seed, Xtr.iloc[a], y[a] / scale[a], scale[a] ** 2 / norm,
                           Xtr.iloc[s], y[s] / scale[s], scale[s] ** 2 / norm, rounds)
        p_val = np.asarray(gbdt_predict(model, m, best, Xtr.iloc[v]), dtype='float64') * scale[v]
        p_test = np.asarray(gbdt_predict(model, m, best, Xt), dtype='float64') * scale_t
        return p_val, p_test, best


    def fit_full_scaled(model, seed, rounds, test_rows=None):
        """全部训练行、固定轮数、不早停的相对力量目标模型。返回测试预测。"""
        Xtr, Xte, _ = FRAMES[model]
        Xt = Xte if test_rows is None else Xte.iloc[test_rows]
        scale, scale_test = SCALE_OF[model]
        scale_t = scale_test if test_rows is None else scale_test[test_rows]
        m, _ = gbdt_fit(model, seed, Xtr, y / scale, scale ** 2 / (scale ** 2).mean(), rounds=rounds)
        return np.asarray(gbdt_predict(model, m, rounds, Xt), dtype='float64') * scale_t


    X_BASE, XT_BASE, CATS = FRAMES['lgb']                # 与 13 相同：LightGBM / TabM 的 ALL3 特征
    NUMS = [c for c in X_BASE.columns if c not in CATS]


    def numeric_with_codes(enc, X):
        return np.hstack([X[NUMS].to_numpy(dtype='float64'), enc.transform(X[CATS])])


    def fit_hgb(seed, rows, v, Xt, small=False):
        """与 13 相同：rows 上训练（内部留 10% 早停）。返回 (v 的预测或 None, 测试预测, 实际轮数)。"""
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan, encoded_missing_value=np.nan,
                             max_categories=250).fit(X_BASE.iloc[rows][CATS])
        model = HistGradientBoostingRegressor(
            categorical_features=[False] * len(NUMS) + [True] * len(CATS), learning_rate=0.05,
            max_iter=20 if small else 3000, max_leaf_nodes=63, min_samples_leaf=40, l2_regularization=1.0,
            early_stopping=True, validation_fraction=0.1, n_iter_no_change=100, random_state=seed)
        model.fit(numeric_with_codes(enc, X_BASE.iloc[rows]), y[rows])
        p_val = None if v is None else np.asarray(model.predict(numeric_with_codes(enc, X_BASE.iloc[v])), dtype='float64')
        return p_val, np.asarray(model.predict(numeric_with_codes(enc, Xt)), dtype='float64'), int(model.n_iter_)


    def folds_for(seed):
        ids = np.zeros(n_train, dtype=int)
        for k, (_, valid) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=seed).split(np.arange(n_train)), start=1):
            ids[valid] = k
        return ids


    def save_result(stem, arrays, info):
        np.savez(OUTPUT_DIR / f'{stem}.npz', **arrays)
        info = {**info, 'gpu': GPU_NAME, 'versions': VERSIONS, 'finished_at_utc': datetime.now(timezone.utc).isoformat()}
        (OUTPUT_DIR / f'{stem}.json').write_text(json.dumps(info, ensure_ascii=False, indent=1), encoding='utf-8')
        return info


    def cached(stem):
        json_path, npz_path = OUTPUT_DIR / f'{stem}.json', OUTPUT_DIR / f'{stem}.npz'
        if json_path.is_file() and npz_path.is_file():
            return json.loads(json_path.read_text(encoding='utf-8'))
        return None


    require(np.array_equal(folds_for(SEED), FOLD_IDS), '种子 42 的五折与之前不一致')
    rng = np.random.default_rng(SEED)
    probe = rng.choice(n_train, 3000, replace=False)
    pa, ps, pv = probe[:2000], probe[2000:2500], probe[2500:]
    OK = {}
    try:
        p_val, p_test, _ = fit_matrix('cb_gl', pa, ps, pv, SEED, rounds=30, test_rows=np.arange(300))
        p_full = with_cb(CB_PARAMS_LR03, fit_full_scaled, 'cb_gl', SEED, 30, test_rows=np.arange(300))
        require(np.isfinite(p_val).all() and np.isfinite(p_test).all() and np.isfinite(p_full).all(), '预测无效')
        OK['cb_gl'] = True
    except Exception as exc:
        OK['cb_gl'] = False
        print('cb_gl 预检失败，之后只用 14 的种子 42 结果：', repr(exc))
    try:
        p_val, p_test, _ = fit_hgb(SEED, np.concatenate([pa, ps]), pv, XT_BASE.iloc[:300], small=True)
        require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), '预测无效')
        OK['hgb'] = True
    except Exception as exc:
        OK['hgb'] = False
        print('HGB 预检失败，之后只用 13 的种子 42 结果：', repr(exc))
    try:
        p_val, p_test = tabm_fit_predict(SEED, pa, ps, pv, XT_NN.iloc[:300], overrides={'n_epochs': 3})
        require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), '预测无效')
        OK['tabm'] = True
    except Exception as exc:
        OK['tabm'] = False
        print('TabM 预检失败，TabM 只用 12 的 3 个 95% 重训：', repr(exc))
    print('预检：', OK)


    def adopt_previous(stem_prev_dir, prev_stem, new_stem):
        """把之前保存的种子 42 五折结果复制到本目录（统一命名）。"""
        for k in range(1, N_FOLDS + 1):
            src = stem_prev_dir / f'{prev_stem}__fold{k}.npz'
            if cached(f'{new_stem}__fold{k}') is None and src.is_file():
                saved = np.load(src)
                src_json = src.with_suffix('.json')
                info = json.loads(src_json.read_text(encoding='utf-8')) if src_json.is_file() else {}
                save_result(f'{new_stem}__fold{k}', {'valid_idx': saved['valid_idx'], 'val': saved['val'],
                                                     'test': saved['test']},
                            {'source': str(src), 'rmse': rmse(y[saved['valid_idx']], saved['val']),
                             'best_iteration': info.get('best_iteration')})


    adopt_previous(ROUND4_DIR, 'cb_gl__R3', 'cb_gl__s42')
    adopt_previous(COMPARE_DIR, 'hgb__s42', 'hgb__s42')


    def run_cv(model, seed):
        ids = folds_for(seed)
        for k in range(1, N_FOLDS + 1):
            stem = f'{model}__s{seed}__fold{k}'
            info = cached(stem)
            if info is None:
                fit_idx, valid_idx = np.flatnonzero(ids != k), np.flatnonzero(ids == k)
                inner_fit, inner_stop = train_test_split(fit_idx, test_size=INNER_STOP_FRACTION, random_state=seed)
                started = time.perf_counter()
                if model == 'cb_gl':
                    p_val, p_test, best = fit_matrix('cb_gl', inner_fit, inner_stop, valid_idx, seed)
                else:
                    p_val, p_test, best = fit_hgb(seed, fit_idx, valid_idx, XT_BASE)
                require(np.isfinite(p_val).all() and np.isfinite(p_test).all(), f'{stem} 预测含非有限值')
                info = save_result(stem, {'valid_idx': valid_idx, 'val': p_val, 'test': p_test},
                                   {'model': model, 'seed': seed, 'fold': k, 'rmse': rmse(y[valid_idx], p_val),
                                    'best_iteration': best, 'seconds': time.perf_counter() - started})
                how = f'{info["seconds"]:.0f}s'
            else:
                how = 'cache'
            print(f'{model:6s} 种子 {seed:4d} fold {k}  RMSE {info["rmse"]:.3f}  ({how})')


    for model in ['hgb', 'cb_gl']:                       # HGB 在 CPU 上很快，先跑
        if not OK.get(model):
            continue
        for seed in PREV_SEEDS:
            try:
                run_cv(model, seed)
            except Exception as exc:
                print(f'!! {model} 种子 {seed} 出错，跳过：{exc!r}')


    def mean_best_iterations(model):
        bests = []
        for seed in PREV_SEEDS:
            for k in range(1, N_FOLDS + 1):
                info = cached(f'{model}__s{seed}__fold{k}')
                if info is not None and info.get('best_iteration'):
                    bests.append(info['best_iteration'])
        return float(np.mean(bests)) if bests else None


    try:                                             # 12 中 cb_rel：学习率 0.03 的最佳轮数约为 0.05 的 1.4 倍
        lr_check = pd.read_csv(FINAL_V2 / 'summary_lr_check.csv')
        LR03_RATIO = float(lr_check['best_iter_0.03'].mean() / lr_check['best_iter_0.05'].mean())
    except Exception:
        LR03_RATIO = 1.44
    print(f'学习率 0.03 / 0.05 的轮数比：{LR03_RATIO:.2f}')

    if OK.get('cb_gl') and mean_best_iterations('cb_gl'):
        gl_rounds = int(round(mean_best_iterations('cb_gl') * LR03_RATIO * REFIT_FACTOR))
        for seed in GL_FULL_SEEDS:
            stem = f'cb_gl__s{seed}__full'
            try:
                info = cached(stem)
                if info is None:
                    started = time.perf_counter()
                    p_test = with_cb(CB_PARAMS_LR03, fit_full_scaled, 'cb_gl', seed, gl_rounds)
                    require(np.isfinite(p_test).all(), f'{stem} 预测含非有限值')
                    info = save_result(stem, {'test': p_test}, {'model': 'cb_gl', 'seed': seed, 'rounds': gl_rounds,
                                                                'learning_rate': 0.03,
                                                                'seconds': time.perf_counter() - started})
                print(f'cb_gl 全量重训 种子 {seed}：轮数 {info["rounds"]}，用时 {info["seconds"]:.0f}s')
            except Exception as exc:
                print(f'!! cb_gl 全量重训 种子 {seed} 出错，跳过：{exc!r}')

    if OK.get('hgb'):
        for seed in HGB_FULL_SEEDS:
            stem = f'hgb__s{seed}__nearfull'
            try:
                info = cached(stem)
                if info is None:
                    started = time.perf_counter()
                    _, p_test, n_iter = fit_hgb(seed, np.arange(n_train), None, XT_BASE)
                    info = save_result(stem, {'test': p_test}, {'model': 'hgb', 'seed': seed, 'n_iter': n_iter,
                                                                'seconds': time.perf_counter() - started})
                print(f'HGB 全部数据（内部 10% 早停）种子 {seed}：轮数 {info["n_iter"]}，用时 {info["seconds"]:.0f}s')
            except Exception as exc:
                print(f'!! HGB 全量 种子 {seed} 出错，跳过：{exc!r}')

    if OK.get('tabm'):
        for seed in TABM_EXTRA_SEEDS:
            stem = f'tabm__s{seed}__nearfull'
            try:
                info = cached(stem)
                if info is None:
                    fit_rows, stop_rows = train_test_split(np.arange(n_train), test_size=TABM_HOLDOUT, random_state=seed)
                    started = time.perf_counter()
                    p_stop, p_test = tabm_fit_predict(seed, fit_rows, stop_rows, stop_rows, XT_NN)
                    require(np.isfinite(p_test).all(), f'{stem} 预测含非有限值')
                    info = save_result(stem, {'test': p_test}, {'model': 'tabm', 'seed': seed,
                                                                'holdout_rmse': rmse(y[stop_rows], p_stop),
                                                                'seconds': time.perf_counter() - started})
                print(f'TabM 95% 重训 种子 {seed}：用时 {info["seconds"]:.0f}s')
            except Exception as exc:
                print(f'!! TabM 种子 {seed} 出错，跳过：{exc!r}')


    def load_cv(directory, stem):
        oof, test_pred = np.full(n_train, np.nan), np.zeros(len(test))
        for k in range(1, N_FOLDS + 1):
            path = directory / f'{stem}__fold{k}.npz'
            if not path.is_file():
                return None
            saved = np.load(path)
            oof[saved['valid_idx']] = saved['val']
            test_pred += saved['test'] / N_FOLDS
        return (oof, test_pred) if not np.isnan(oof).any() else None


    def full_files(model):
        if model in ('lgb', 'xgb', 'lgb_rel', 'catboost', 'cb_rel'):
            return sorted(FINAL_V1.glob(f'{model}__s*__full.npz')) + sorted(FINAL_V2.glob(f'{model}__s*__full.npz'))
        if model == 'tabm':
            return sorted(FINAL_V2.glob('tabm__s*__nearfull.npz')) + sorted(OUTPUT_DIR.glob('tabm__s*__nearfull.npz'))
        if model == 'hgb':
            return sorted(OUTPUT_DIR.glob('hgb__s*__nearfull.npz'))
        return sorted(OUTPUT_DIR.glob(f'{model}__s*__full.npz'))


    ALL_MODELS = ['lgb', 'xgb', 'lgb_rel', 'catboost', 'cb_rel', 'tabm', 'hgb', 'cb_gl']
    PER, rows = {}, []
    for model in ALL_MODELS:
        directory = FINAL_V1 if model in FINAL_MODELS else OUTPUT_DIR
        loaded = {s: load_cv(directory, f'{model}__s{s}') for s in PREV_SEEDS}
        loaded = {s: v for s, v in loaded.items() if v is not None}
        if not loaded:
            print(f'{model}：没有可用的交叉验证结果，跳过')
            continue
        oof = np.mean([o for o, _ in loaded.values()], axis=0)
        fold_avg = np.mean([t for _, t in loaded.values()], axis=0)
        files = full_files(model)
        full = np.mean([np.load(p)['test'] for p in files], axis=0) if files else fold_avg
        PER[model] = {'oof': oof, 'full': full}
        rows.append({'model': model, 'cv_seeds': list(loaded), 'oof_seed_avg': rmse(y, oof),
                     'test_from': f'{len(files)} 个全量 / 95% 重训' if files else '折平均'})


    def nested_nnls(P):
        out = np.zeros(n_train)
        for k in range(1, N_FOLDS + 1):
            fit, va = FOLD_IDS != k, FOLD_IDS == k
            w, _ = nnls(P[fit], y[fit])
            out[va] = P[va] @ w
        return out


    NAMES = list(PER)
    P = np.column_stack([PER[n]['oof'] for n in NAMES])
    WEIGHTS, _ = nnls(P, y)
    NESTED = nested_nnls(P)
    model_table = pd.DataFrame(rows).set_index('model')
    model_table['weight'] = pd.Series(dict(zip(NAMES, WEIGHTS)))
    display(model_table.round(3))
    model_table.to_csv(OUTPUT_DIR / 'summary_models.csv')

    steps = {'11 的 6 个模型': [m for m in NAMES if m in FINAL_MODELS],
             '+ HGB': [m for m in NAMES if m in FINAL_MODELS + ['hgb']],
             '+ HGB + cb_gl（最终）': NAMES}
    step_rows = [{'融合': k, '模型数': len(v), '嵌套 OOF RMSE': rmse(y, nested_nnls(np.column_stack([PER[n]['oof'] for n in v])))}
                 for k, v in steps.items() if v]
    display(pd.DataFrame(step_rows).set_index('融合').round(3))
    BLEND_TEST = {'v4': np.column_stack([PER[n]['full'] for n in NAMES]) @ WEIGHTS}
    print('NNLS 权重：', dict(zip(NAMES, WEIGHTS.round(3))))


    CORE = CORE9


    def row_keys(frame, columns):
        return pd.util.hash_pandas_object(frame[columns], index=False).to_numpy()


    def mate_stats(train_keys, query_keys=None):
        table = pd.DataFrame({'k': train_keys, 'y': y}).groupby('k')['y'].agg(['sum', 'count'])
        if query_keys is None:                      # 训练行：留一
            keys = pd.Series(train_keys)
            n = keys.map(table['count']).to_numpy() - 1
            s = keys.map(table['sum']).to_numpy() - y
        else:                                       # 测试行：全部训练行
            keys = pd.Series(query_keys)
            n = keys.map(table['count']).fillna(0).to_numpy()
            s = keys.map(table['sum']).fillna(0).to_numpy()
        return n, np.where(n > 0, s / np.maximum(n, 1), np.nan)


    def twins(query_exact=None, query_core=None):
        n_ex, m_ex = mate_stats(row_keys(train, RAW_FEATURES), query_exact)
        n_co, m_co = mate_stats(row_keys(train, CORE), query_core)
        level = np.select([n_ex > 0, n_co > 0], ['exact', 'core'], default='none')
        return level, np.where(level == 'exact', m_ex, m_co)


    def fit_alpha(mask, pred, mate):
        d = mate[mask] - pred[mask]
        denom = float(np.sum(d * d))
        return float(np.clip(np.sum((y[mask] - pred[mask]) * d) / denom, 0, 1)) if denom > 0 else 0.0


    def apply_twins(pred, mate, level, alphas):
        out = pred.copy()
        for lv, a in alphas.items():
            m = level == lv
            out[m] = pred[m] + a * (mate[m] - pred[m])
        return out


    LEVEL, MATE = twins()
    T_LEVEL, T_MATE = twins(row_keys(test, RAW_FEATURES), row_keys(test, CORE))
    HALF = (pd.util.hash_pandas_object(train[['Sex', 'Country', 'State']], index=False).to_numpy() % 2).astype(int)
    TR, TE = train.reset_index(drop=True), test.reset_index(drop=True)


    def text_key(frame, columns):
        out = frame[columns[0]].astype(str)
        for c in columns[1:]:
            out = out + '|' + frame[c].astype(str)
        return out


    BY_TR = np.floor(TR['MeetYear'] - TR['Age'] + 1e-9).astype(int)
    BY_TE = np.floor(TE['MeetYear'] - TE['Age'] + 1e-9).astype(int)
    YR_TR, YR_TE = TR['MeetYear'].astype(int), TE['MeetYear'].astype(int)
    PREFIX = {'A0': ['Sex', 'Country', 'State'], 'A1': ['Sex', 'Country', 'State', 'Federation'],
              'A2': ['Sex', 'Country', 'State', 'Federation', 'Equipment'],
              'A3': ['Sex', 'Country', 'State', 'Federation', 'Equipment', 'WeightClassKg']}
    SHRINK = {'A0': 5.0, 'A1': 5.0, 'A2': 3.0, 'A3': 2.0}
    PFX_TR = {k: text_key(TR, c) for k, c in PREFIX.items()}
    PFX_TE = {k: text_key(TE, c) for k, c in PREFIX.items()}
    MEET_COLS = ['Federation', 'MeetYear', 'MeetCountry', 'MeetState', 'Sex', 'Equipment']
    MEET_TR, MEET_TE = text_key(TR, MEET_COLS), text_key(TE, MEET_COLS)
    VALID_TR = (TR['Country'] != 'Unknown').to_numpy(dtype='float64')
    VALID_TE = (TE['Country'] != 'Unknown').to_numpy(dtype='float64')


    def sums(keys, values):
        table = pd.DataFrame({'k': keys.to_numpy(), 'v': values}).groupby('k')['v'].agg(['sum', 'count'])
        return table['sum'], table['count']


    def look(table, keys):
        s, n = table
        return keys.map(s).fillna(0).to_numpy(dtype='float64'), keys.map(n).fillna(0).to_numpy(dtype='float64')


    def f_exact(k, r):
        ktr, kte = PFX_TR[k] + '|' + BY_TR.astype(str), PFX_TE[k] + '|' + BY_TE.astype(str)
        table = sums(ktr, r)
        s, n = look(table, ktr)
        st, nt = look(table, kte)
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


    def f_fuzzy(k, r):
        table = sums(PFX_TR[k] + '|' + BY_TR.astype(str), r)
        s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
        for d in (-1, 0, 1):
            a, b = look(table, PFX_TR[k] + '|' + (BY_TR + d).astype(str))
            s, n = s + a, n + b
            a, b = look(table, PFX_TE[k] + '|' + (BY_TE + d).astype(str))
            st, nt = st + a, nt + b
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])


    def f_decay(k, r, tau=3.0, max_gap=8):
        base_tr = PFX_TR[k] + '|' + BY_TR.astype(str) + '|'
        base_te = PFX_TE[k] + '|' + BY_TE.astype(str) + '|'
        table = sums(base_tr + YR_TR.astype(str), r)
        s, n, st, nt = np.zeros(n_train), np.zeros(n_train), np.zeros(len(TE)), np.zeros(len(TE))
        for d in range(-max_gap, max_gap + 1):
            w = float(np.exp(-abs(d) / tau))
            a, b = look(table, base_tr + (YR_TR + d).astype(str))
            s, n = s + w * a, n + w * b
            a, b = look(table, base_te + (YR_TE + d).astype(str))
            st, nt = st + w * a, nt + w * b
        return (s - r) / (n - 1 + SHRINK[k]), st / (nt + SHRINK[k])     # d = 0 时自己权重为 1，已扣除


    def f_meet(r, lam=20.0):
        table = sums(MEET_TR, r)
        s, n = look(table, MEET_TR)
        st, nt = look(table, MEET_TE)
        return (s - r) / (n - 1 + lam), st / (nt + lam)


    def s3_features(r):
        cols_tr, cols_te = [], []
        for builder in (f_exact, f_fuzzy, f_decay):
            for k in PREFIX:
                a, b = builder(k, r)
                cols_tr.append(a * VALID_TR)
                cols_te.append(b * VALID_TE)
        a, b = f_meet(r)
        cols_tr.append(a)
        cols_te.append(b)
        return np.column_stack(cols_tr), np.column_stack(cols_te)


    # 第二层只依赖训练行的 OOF（嵌套融合），对三种测试预测使用同一组 alpha / 系数
    alphas = {lv: fit_alpha(LEVEL == lv, NESTED, MATE) for lv in ['exact', 'core']}
    p1 = apply_twins(NESTED, MATE, LEVEL, alphas)
    p1_cf = NESTED.copy()
    for h in (0, 1):
        a_h = {lv: fit_alpha((HALF != h) & (LEVEL == lv), NESTED, MATE) for lv in ['exact', 'core']}
        m = HALF == h
        p1_cf[m] = apply_twins(NESTED[m], MATE[m], LEVEL[m], a_h)
    r1 = y - p1
    F, FT = s3_features(r1)
    p2_cf = p1_cf.copy()
    for h in (0, 1):
        fit, m = HALF != h, HALF == h
        beta_h = np.linalg.lstsq(F[fit], r1[fit], rcond=None)[0]
        p2_cf[m] = p1_cf[m] + F[m] @ beta_h
    BETA = np.linalg.lstsq(F, r1, rcond=None)[0]
    STAGE2_ADJ = FT @ BETA
    stage_table = pd.DataFrame([{'step': '融合（嵌套）', 'oof_rmse': rmse(y, NESTED)},
                                {'step': '+ 双胞胎（交叉拟合）', 'oof_rmse': rmse(y, p1_cf)},
                                {'step': '+ S3 平滑（交叉拟合）', 'oof_rmse': rmse(y, p2_cf)}]).set_index('step')
    display(stage_table.round(3))
    stage_table.to_csv(OUTPUT_DIR / 'summary_stage2.csv')
    print('alpha：', {k: round(v, 3) for k, v in alphas.items()}, '｜测试中有双胞胎的行：', int((T_LEVEL != 'none').sum()),
          '｜S3 平均调整幅度：', round(float(np.abs(STAGE2_ADJ).mean()), 2), 'kg')

    for kind, pred in BLEND_TEST.items():
        final = apply_twins(pred, T_MATE, T_LEVEL, alphas) + STAGE2_ADJ
        for stage, values in [('base', pred), ('final', final)]:
            path = OUTPUT_DIR / f'submission_final_{kind}_{stage}.csv'
            pd.DataFrame({'id': test['id'], TARGET: np.clip(values, 0, None)}).to_csv(path, index=False)
            print('写出', path.name)
    np.save(OUTPUT_DIR / 'oof_nested_blend.npy', NESTED)
    np.save(OUTPUT_DIR / 'oof_final_crossfit.npy', p2_cf)


    sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
    final_path = OUTPUT_DIR / 'submission_final_v4_final.csv'
    final = pd.read_csv(final_path)
    require(final.columns.tolist() == ['id', TARGET], '列名应为 id, TotalKg')
    require(len(final) == len(sample) == 45000 and final['id'].is_unique
            and set(final['id']) == set(sample['id']), 'id 与 sample_submission 不一致')   # Kaggle 按 id 匹配，顺序无关
    require(np.isfinite(final[TARGET]).all() and (final[TARGET] > 0).all(), '预测含无效值')
    print(f'检查通过：{final_path.name}，{len(final)} 行；预测均值 {final[TARGET].mean():.1f}，'
          f'范围 {final[TARGET].min():.1f}–{final[TARGET].max():.1f}（训练集均值 {y.mean():.1f}）')
    previous = COMPARE_DIR / 'submission_final_v3_plus_final.csv'
    if previous.is_file():
        prev = pd.read_csv(previous)
        diff = final[TARGET].to_numpy() - prev[TARGET].to_numpy()
        print(f'与当前公开榜最好的 v3_plus（76.334）相比：相关 {np.corrcoef(final[TARGET], prev[TARGET])[0, 1]:.5f}，'
              f'平均绝对差 {np.abs(diff).mean():.2f} kg，最大 {np.abs(diff).max():.1f} kg')
    print()
    print('==== 最终在 Kaggle 上选择这两个提交 ====')
    print('1.', final_path.relative_to(PROJECT_DIR), '（本 notebook：8 个模型 + 双胞胎 + S3，CV 最好）')
    print('2.', previous.relative_to(PROJECT_DIR) if previous.is_file() else '目前公开榜最好的提交',
          '（目前公开榜最好的 76.334，作为保险）')
    print('如果第 1 个在公开榜上明显变差（比 76.334 差 0.05 以上），改选 v3_plus 和', (FINAL_V2 / 'submission_final_v2_seeds_tabmfull_final.csv').relative_to(PROJECT_DIR))
else:
    print("训练开关关闭（TRAIN['eight_models'] = False）：本节只读取 artifacts/ 中已保存的结果。")

In [ ]:
def show_v4():
    """从保存的逐折预测重新计算实验 15 的融合，并与保存的提交文件核对（只读，不训练）。"""
    v1 = ARTIFACTS / '04_final_model' / 'v1_six_models'
    v2 = ARTIFACTS / '04_final_model' / 'v2_more_seeds'
    v4 = ARTIFACTS / '04_final_model' / 'v4_eight_models'
    six = ['lgb', 'xgb', 'lgb_rel', 'catboost', 'cb_rel', 'tabm']
    seeds = [42, 7, 2024]

    def refit_files(model):
        if model in ('lgb', 'xgb', 'lgb_rel', 'catboost', 'cb_rel'):
            return sorted(v1.glob(f'{model}__s*__full.npz')) + sorted(v2.glob(f'{model}__s*__full.npz'))
        if model == 'tabm':
            return sorted(v2.glob('tabm__s*__nearfull.npz')) + sorted(v4.glob('tabm__s*__nearfull.npz'))
        if model == 'hgb':
            return sorted(v4.glob('hgb__s*__nearfull.npz'))
        return sorted(v4.glob(f'{model}__s*__full.npz'))

    per, rows = {}, []
    for model in six + ['hgb', 'cb_gl']:
        loaded = {s: read_folds(v1 if model in six else v4, f'{model}__s{s}') for s in seeds}
        loaded = {s: v for s, v in loaded.items() if v is not None}
        if not loaded:
            print(f'{model}：缺少五折结果，跳过')
            continue
        oof = np.mean([o for o, _ in loaded.values()], axis=0)
        files = refit_files(model)
        full = (np.mean([np.load(p)['test'] for p in files], axis=0) if files
                else np.mean([t for _, t in loaded.values()], axis=0))
        per[model] = {'oof': oof, 'full': full}
        rows.append({'model': model, **{f'种子 {s}': rmse(y, o) for s, (o, _) in loaded.items()},
                     '三种子平均 OOF': rmse(y, oof),
                     '测试预测': f'{len(files)} 个全量 / 95% 重训' if files else '折平均'})
    if not per:
        return
    names = list(per)
    weights, _ = nnls(np.column_stack([per[n]['oof'] for n in names]), y)
    table = pd.DataFrame(rows).set_index('model')
    table['NNLS 权重'] = pd.Series(dict(zip(names, weights)))
    display(table.round(3))

    def nested(columns):
        P_sub = np.column_stack([per[n]['oof'] for n in columns])
        out = np.zeros(n_train)
        for k in range(1, N_FOLDS + 1):
            fit, va = FOLD_IDS != k, FOLD_IDS == k
            w, _ = nnls(P_sub[fit], y[fit])
            out[va] = P_sub[va] @ w
        return out

    steps = {'实验 11：6 个模型': [m for m in names if m in six],
             '+ HGB（实验 13）': [m for m in names if m in six + ['hgb']],
             '+ HGB + cb_gl（最终，实验 15）': names}
    display(pd.DataFrame([{'融合': k, '模型数': len(v), '嵌套 OOF RMSE': rmse(y, nested(v))}
                          for k, v in steps.items() if v]).set_index('融合').round(3))
    print('第二层（交叉拟合）：')
    show(read_summary(v4 / 'summary_stage2.csv'))

    blend_test = np.clip(np.column_stack([per[n]['full'] for n in names]) @ weights, 0, None)
    saved_path = v4 / 'submission_final_v4_base.csv'
    if saved_path.is_file() and len(names) == 8:
        saved = pd.read_csv(saved_path)
        require(saved['id'].equals(test['id']), f'{saved_path.name} 的 id 顺序与 test.csv 不同')
        print(f'重新计算的融合测试预测与保存的 {saved_path.name} 最大相差 '
              f'{np.abs(blend_test - saved[TARGET].to_numpy()).max():.2e} kg')


show_v4()

**结论**：
- cb_gl（三种子 76.781）与 cb_rel（76.784）一样强，权重最大（0.29）。
- HGB 单独较弱，但错误与其他模型不同，权重 0.20。
- LightGBM、lgb_rel、CatBoost 的权重变为 0，它们的信息已被 cb_rel / cb_gl / HGB 覆盖。
- 嵌套融合 76.395 → 76.351（+ HGB）→ **76.319**（+ cb_gl）；加第二层后 **75.966**，是全部实验中最好的交叉验证分数。

## 5. 提交文件与公开榜

公开榜只用约 30% 的测试数据（约 13,500 行），0.05 kg 以内的差异基本是噪声（05），所以选模以交叉验证为准。全量重训的版本没有对应的交叉验证分数。

In [ ]:
final_file = ARTIFACTS / '04_final_model' / 'v4_eight_models' / 'submission_final_v4_final.csv'
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')
if final_file.is_file():
    final_sub = pd.read_csv(final_file)
    require(final_sub.columns.tolist() == ['id', TARGET], '列名应为 id, TotalKg')
    require(len(final_sub) == len(sample_submission) == 45000 and final_sub['id'].is_unique
            and set(final_sub['id']) == set(sample_submission['id']), 'id 与 sample_submission 不一致')
    require(np.isfinite(final_sub[TARGET]).all() and (final_sub[TARGET] > 0).all(), '预测含无效值')
    print(f'检查通过：{final_file.relative_to(PROJECT_DIR)}，{len(final_sub)} 行；预测均值 {final_sub[TARGET].mean():.1f}，'
          f'范围 {final_sub[TARGET].min():.1f}–{final_sub[TARGET].max():.1f}（训练集均值 {y.mean():.1f}）')
else:
    print(f'缺少 {final_file.relative_to(PROJECT_DIR)}')

leaderboard = pd.DataFrame([
    ('11', '6 个模型 × 3 种子，折平均 + 双胞胎 + S3', 76.044, 76.520),
    ('11', '同上，树模型全量重训（混合版 76.421）', np.nan, 76.360),
    ('12', '全量重训共 8 个种子', np.nan, 76.359),
    ('12', '+ TabM 95% 重训 × 3 种子', np.nan, 76.337),
    ('13', 'v3_plus：+ sklearn HGB', 76.010, 76.334),
    ('15', 'v4：8 个模型（+ cb_gl、HGB 三种子，TabM 重训 5 个种子）', 75.966, np.nan),
], columns=['实验', '提交', 'CV（嵌套融合 + 交叉拟合第二层）', '公开榜'])
display(leaderboard.set_index('实验'))

**Kaggle 最终选择两个提交**（私榜按另外 70% 的测试数据计算）：

1. `artifacts/04_final_model/v4_eight_models/submission_final_v4_final.csv`：交叉验证最好，主选；
2. `artifacts/03_model_selection/model_compare/submission_final_v3_plus_final.csv`：公开榜最好（76.334），作保险。

如果 v4 在公开榜上比 76.334 差 0.05 以上，改选 v3_plus 和 `artifacts/04_final_model/v2_more_seeds/submission_final_v2_seeds_tabmfull_final.csv`。三个文件的副本都在 `submissions/`。